### 가설2 더 강건한 검정

In [35]:
# # 8. 업종 정보 병합
# if '기업명' in val_df.columns and '기업명' in industry_map.columns:
#     val_df = val_df.merge(industry_map, on='기업명', how='left')


np.int64(440290)

In [45]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler
from scipy.stats.mstats import winsorize
from statsmodels.tools import add_constant
import warnings
import re
from scipy import stats

warnings.filterwarnings('ignore')

# ===============================
# 종속변수 설정
# ===============================
DEPENDENT_VAR = 'MBV' # 'log_MBV' # 로그 변환된 MBV 사용
print(f"=== 선택된 종속변수: {DEPENDENT_VAR} (로그 변환) ===")

# ===============================
# 1. 데이터 로드 및 기본 정보 파악
# ===============================
print("=== 데이터 로드 및 탐색적 분석 ===")

# 데이터 로드
low_val = pd.read_csv("/content/회계정보하위권_기업가치및재무정보.csv")
high_val = pd.read_csv("/content/회계정보상위권_기업가치및재무정보.csv")
low_news = pd.read_csv("/content/최종검정용뉴스감성(하위권).csv")
high_news = pd.read_csv("/content/최종검정용뉴스감성(상위권).csv")
industry_map = pd.read_excel("/content/최종가설검정용_기업들산업군매핑표.xlsx")

# 산업 매핑 테이블 정리
industry_map.columns = industry_map.columns.str.strip()
print(industry_map.columns.unique())
if '최종 산업군' in industry_map.columns:
  industry_map = industry_map.rename(columns={"최종 산업군": "industry"})
elif '산업군' in industry_map.columns:
  industry_map = industry_map.rename(columns={"산업군": "industry"})
elif '산업' in industry_map.columns:
  industry_map = industry_map.rename(columns={"산업": "industry"})
else:
  cols = industry_map.columns.tolist()
  if len(cols) >= 2:
    industry_map = industry_map.rename(columns={cols[1]: "industry"})

print("\n=== 뉴스 감성 데이터 분포 탐색 ===")
print(f"하위권 뉴스: {len(low_news)}건, 상위권 뉴스: {len(high_news)}건")

# ===============================
# 2. 데이터 전처리 (최소한만)
# ===============================
print("\n=== 데이터 전처리 ===")

def smart_preprocessing(val_df, group_name):
  """스마트한 전처리 - 표본 크기 보존 우선"""
  initial_count = len(val_df)

  # 1. 기본 필터링만
  val_df = val_df[val_df['market_cap_krw'] > 0]
  val_df = val_df[val_df['자산총계'] > 0]
  val_df = val_df[val_df['자본총계'] > 0]

  # 2. MBV 계산
  val_df['MBV'] = val_df['market_cap_krw'] / val_df['자본총계']

  # 3. 극단적인 이상치만 제거 (상하위 1%만)
  mbv_q01 = val_df['MBV'].quantile(0.01)
  mbv_q99 = val_df['MBV'].quantile(0.99)
  val_df = val_df[(val_df['MBV'] >= mbv_q01) & (val_df['MBV'] <= mbv_q99)]

  # 4. 로그 MBV 계산
  val_df['log_MBV'] = np.log(val_df['MBV'])

  # 5. 재무비율 계산
  val_df['leverage'] = val_df['부채총계'] / val_df['자산총계']
  val_df['roa'] = val_df['당기순이익'] / val_df['자산총계']
  val_df['roe'] = val_df['당기순이익'] / val_df['자본총계']

  if '매출액' in val_df.columns:
    # 매출 0도 허용 (서비스업 등 고려)
    val_df['asset_turnover'] = np.where(val_df['매출액'] > 0,
                     val_df['매출액'] / val_df['자산총계'], 0)
    val_df['profit_margin'] = np.where(val_df['매출액'] > 0,
                    val_df['당기순이익'] / val_df['매출액'], np.nan)

  # 6. 무한대값 처리
  for col in ['leverage', 'roa', 'roe', 'asset_turnover', 'profit_margin']:
    if col in val_df.columns:
      val_df[col] = val_df[col].replace([np.inf, -np.inf], np.nan)

  # 7. 기업명 정리
  if '기업명' in val_df.columns:
    val_df['기업명'] = val_df['기업명'].astype(str).str.strip()

  # 8. 업종 정보 병합
  if '기업명' in val_df.columns and '기업명' in industry_map.columns:
    val_df = val_df.merge(industry_map, on='종목코드', how='left')
    val_df['industry'] = val_df['industry'].fillna('기타')

  final_count = len(val_df)
  print(f"{group_name} - 전처리 전: {initial_count}, 후: {final_count} (제거율: {((initial_count-final_count)/initial_count*100):.1f}%)")

  return val_df

low_val_processed = smart_preprocessing(low_val, "하위권")
high_val_processed = smart_preprocessing(high_val, "상위권")

# ===============================
# 3. 뉴스 데이터 집계 (최소 기준 낮춤 + 뉴스 개수 통제변수로 활용)
# ===============================
print("\n=== 뉴스 데이터 집계 (뉴스 개수를 통제변수로 활용) ===")

def flexible_news_aggregation(news_df, group_name, min_news=2):
  """유연한 뉴스 집계 - 뉴스 개수를 통제변수로 활용"""

  # 1. 최소 뉴스 개수 기준 완화 (2개 이상)
  news_count = news_df.groupby(['종목코드', '종목명', '기사년도']).size()
  valid_groups = news_count[news_count >= min_news].index
  news_filtered = news_df.set_index(['종목코드', '종목명', '기사년도']).loc[valid_groups].reset_index()

  print(f"{group_name} - 필터링 전: {len(news_df)}건  후: {len(news_filtered)}건")
  print(f"유효 기업-연도 조합: {len(valid_groups)}개")

  # 2. 집계 계산
  agg_df = news_filtered.groupby(['종목코드', '종목명', '기사년도']).agg({
    '재조정_긍정확률_평균': ['mean', 'std', 'count', 'min', 'max'],
    '재조정_부정확률_평균': ['mean', 'std']
  }).round(4)

  agg_df.columns = ['positive_mean', 'positive_std', 'news_count', 'positive_min', 'positive_max',
          'negative_mean', 'negative_std']
  agg_df = agg_df.reset_index()

  # 3. 감성 지표 다양화
  agg_df['sentiment_score'] = agg_df['positive_mean'] - agg_df['negative_mean']
  agg_df['sentiment_volatility'] = agg_df['positive_std'].fillna(0)
  agg_df['sentiment_range'] = agg_df['positive_max'] - agg_df['positive_min']

  # 4. 뉴스 관련 변수들 (통제변수로 활용)
  agg_df['log_news_count'] = np.log(agg_df['news_count'])
  agg_df['news_intensity'] = agg_df['news_count'] / agg_df['news_count'].max() # 정규화된 뉴스 강도

  # 5. 뉴스 품질 지표
  agg_df['news_quality'] = np.where(agg_df['news_count'] >= 5, 1, 0) # 고품질 더미
  agg_df['news_coverage'] = pd.qcut(agg_df['news_count'], q=3, labels=['Low', 'Medium', 'High']) # 보도량 구간

  print(f"{group_name} - 최종 기업수: {agg_df['종목명'].nunique()}, 관측치: {len(agg_df)}")
  print(f"뉴스 개수 분포 - 평균: {agg_df['news_count'].mean():.1f}, 중위수: {agg_df['news_count'].median():.0f}")


  return agg_df


low_news_agg = flexible_news_aggregation(low_news, "하위권", min_news=2)
high_news_agg = flexible_news_aggregation(high_news, "상위권", min_news=2)


# ===============================
# 4. 데이터 병합
# ===============================
print("\n=== 데이터 병합 ===")


# 병합
low_merged = pd.merge(
  low_val_processed, low_news_agg,
  left_on=['종목코드', 'bsns_year'],
  right_on=['종목코드', '기사년도'],
  how='inner'
)
low_merged['info_group'] = 1 # 하위권


high_merged = pd.merge(
  high_val_processed, high_news_agg,
  left_on=['종목코드', 'bsns_year'],
  right_on=['종목코드', '기사년도'],
  how='inner'
)
high_merged['info_group'] = 0 # 상위권


# 전체 데이터셋
full_df = pd.concat([low_merged, high_merged], ignore_index=True)


print(f"병합 후 전체 데이터: {full_df.shape}")
print(f"하위권: {(full_df['info_group'] == 1).sum()}개, 상위권: {(full_df['info_group'] == 0).sum()}개")


# ===============================
# 5. 변수 생성 및 변환
# ===============================
print("\n=== 변수 생성 및 변환 ===")


# 필요 변수 선택
analysis_vars = [
  'log_MBV', 'MBV', 'positive_mean', 'sentiment_score', 'sentiment_volatility',
  'news_count', 'log_news_count', 'news_intensity', 'news_quality', 'news_coverage',
  'info_group', 'bsns_year', '자산총계', 'leverage', 'roa', 'roe', 'industry'
]


existing_vars = [var for var in analysis_vars if var in full_df.columns]
final_df = full_df[existing_vars].copy()


# 컬럼명 변경
rename_dict = {
  'positive_mean': 'sentiment_mean',
  'bsns_year': 'year',
  '자산총계': 'total_assets'
}
final_df = final_df.rename(columns=rename_dict)


print("결측치 제거 전:", final_df.shape)
final_df = final_df.dropna()
print("결측치 제거 후:", final_df.shape)


# 추가 변수 생성
if 'total_assets' in final_df.columns:
  final_df['log_assets'] = np.log(final_df['total_assets'])


if 'year' in final_df.columns:
  final_df['year_center'] = final_df['year'] - final_df['year'].mean()


# 뉴스 커버리지 더미변수
if 'news_coverage' in final_df.columns:
  coverage_dummies = pd.get_dummies(final_df['news_coverage'], prefix='coverage', drop_first=True)
  final_df = pd.concat([final_df, coverage_dummies], axis=1)


# 산업 더미 (주요 산업만)


if 'industry' in final_df.columns:
  industry_counts = final_df['industry'].value_counts()
  major_industries = industry_counts[industry_counts >= 5].index
  print("major_industries", major_industries)
  final_df['industry_major'] = final_df['industry'].apply(
    lambda x: x if x in major_industries else '기타'
  )


  # 'industry_major'를 카테고리형으로 변환, '기타'를 첫 번째 레벨로 설정
  final_df['industry_major'] = pd.Categorical(
    final_df['industry_major'],
    categories=['기타'] + [ind for ind in major_industries if ind != '기타'],
    ordered=False
  )


  industry_dummies = pd.get_dummies(final_df['industry_major'], prefix='ind', drop_first=True)
  print("industry_dummies", industry_dummies)


  # 컬럼명 안전화
  def sanitize_colnames(cols):
    new = []
    for c in cols:
      c2 = re.sub(r'\W+', '_', str(c))
      c2 = re.sub(r'_+', '_', c2).strip('_')
      if re.match(r'^[0-9]', c2):
        c2 = 'X_' + c2
      new.append(c2)
    return new


  industry_dummies.columns = sanitize_colnames(industry_dummies.columns.tolist())
  final_df = pd.concat([final_df, industry_dummies], axis=1)


# 표준화
scaler = StandardScaler()
scale_vars = ['sentiment_mean', 'sentiment_score', 'log_assets', 'leverage', 'log_news_count', 'news_intensity']


for v in scale_vars:
  if v in final_df.columns:
    final_df[f'{v}_scaled'] = scaler.fit_transform(final_df[[v]])


print(f"최종 데이터: {final_df.shape}")


# ===============================
# 6. 정규성 확인 및 기술통계
# ===============================
print("\n=== 정규성 확인 및 기술통계 ===")


# 정규성 검정
if 'MBV' in final_df.columns and 'log_MBV' in final_df.columns:
  print("정규성 검정 결과:")


  if len(final_df) <= 50:
    stat_mbv, p_mbv = stats.shapiro(final_df['MBV'])
    stat_log, p_log = stats.shapiro(final_df['log_MBV'])
    print(f"MBV 원본: p-value={p_mbv:.4f}")
    print(f"log_MBV: p-value={p_log:.4f}")
  else:
    print("표본이 커서 Shapiro-Wilk 대신 시각적 확인 권장")


# 기술통계
desc_vars = ['log_MBV', 'sentiment_mean', 'sentiment_score', 'news_count', 'info_group', 'log_assets', 'leverage']
exist_desc = [c for c in desc_vars if c in final_df.columns]


print(f"\n주요 변수들의 기술통계:")
print(final_df[exist_desc].describe().round(3))


if 'info_group' in final_df.columns:
  print("\n그룹별 평균:")
  print(final_df.groupby('info_group')[exist_desc].mean().round(3))


  # 뉴스 개수 분포 확인
  print(f"\n뉴스 개수 그룹별 분포:")
  print(f"하위권 평균: {final_df[final_df['info_group']==1]['news_count'].mean():.1f}")
  print(f"상위권 평균: {final_df[final_df['info_group']==0]['news_count'].mean():.1f}")


# ===============================
# 7. 뉴스 개수를 통제하는 회귀분석
# ===============================
print("\n" + "="*60)
print(f"뉴스 개수 통제 회귀분석 (종속변수: {DEPENDENT_VAR})")
print("="*60)


# 모델들
try:
  # 모델 1: 기본 상호작용 + 뉴스 개수 통제
  formula1 = f"{DEPENDENT_VAR} ~ sentiment_mean_scaled * info_group + log_news_count_scaled + log_assets_scaled + leverage_scaled"
  model1 = smf.ols(formula1, data=final_df).fit()


  print(f"\n[모델 1: 뉴스 개수 통제 기본 모델]")
  print(f"회귀식: {formula1}")
  print(model1.summary())


  # 모델 2: 뉴스 품질 더미 추가
  if 'news_quality' in final_df.columns:
    formula2 = f"{DEPENDENT_VAR} ~ sentiment_mean_scaled * info_group + log_news_count_scaled + news_quality + log_assets_scaled + leverage_scaled"
    model2 = smf.ols(formula2, data=final_df).fit()


    print(f"\n[모델 2: 뉴스 품질 통제]")
    print(model2.summary())
  else:
    model2 = model1


  # 모델 3: 뉴스 강도 통제
  if 'news_intensity_scaled' in final_df.columns:
    formula3 = f"{DEPENDENT_VAR} ~ sentiment_mean_scaled * info_group + news_intensity_scaled + log_assets_scaled + leverage_scaled"
    model3 = smf.ols(formula3, data=final_df).fit()


    print(f"\n[모델 3: 뉴스 강도 통제]")
    print(model3.summary())
  else:
    model3 = model1


  # 모델 4: 업종 효과 추가 (데이터가 충분할 경우)
  industry_terms = []
  for col in final_df.columns:
    if col.startswith('ind_') and final_df[col].sum() >= 3: # 최소 3개 이상
      industry_terms.append(col)


  if industry_terms and len(final_df) > 50: # 충분한 표본이 있을 때만
    formula4 = formula1 + " + " + " + ".join(industry_terms)
    model4 = smf.ols(formula4, data=final_df).fit()


    print(f"\n[모델 4: 업종 효과 포함]")
    print(model4.summary())
  else:
    model4 = model1


  # 결과 요약
  models = [model1, model2, model3, model4]
  model_names = ["뉴스개수통제", "뉴스품질통제", "뉴스강도통제", "업종효과포함"]


  print(f"\n{'='*50}")
  print("상호작용 효과 종합 결과")
  print(f"{'='*50}")


  best_model = None
  best_pvalue = 1.0


  for i, (model, name) in enumerate(zip(models, model_names)):
    try:
      coef = model.params.get('sentiment_mean_scaled:info_group', None)
      pval = model.pvalues.get('sentiment_mean_scaled:info_group', None)
      rsq = model.rsquared
      n_obs = model.nobs


      if coef is not None and pval is not None:
        significance = ""
        if pval < 0.01:
          significance = "***"
        elif pval < 0.05:
          significance = "**"
        elif pval < 0.10:
          significance = "*"


        print(f"{name}: 계수={coef:.4f}{significance} (p={pval:.4f}), R={rsq:.3f}, N={n_obs:.0f}")


        # 최적 모델 선택
        if pval < best_pvalue:
          best_pvalue = pval
          best_model = name


    except Exception as e:
      print(f"{name}: 결과 추출 실패 - {e}")


  print(f"\n최적 모델: {best_model} (p-value: {best_pvalue:.4f})")


  # 가설 검정 결론
  print(f"\n가설 검정 결론:")
  if best_pvalue < 0.05:
    print("H1 채택: 하위권 기업에서 뉴스 감성의 영향이 더 크다 (p < 0.05)")
  elif best_pvalue < 0.10:
    print("H1 약한 채택: 하위권 기업에서 뉴스 감성의 영향이 더 크다 (p < 0.10)")
  else:
    print(f"H0 채택: 그룹간 차이가 통계적으로 유의하지 않다 (p = {best_pvalue:.4f})")


except Exception as e:
  print("회귀분석 실행 오류:", e)


# ===============================
# 8. 추가 분석 제안
# ===============================
print("\n" + "="*60)
print("분석 결과 및 개선 방향")
print("="*60)


print(f"""
현재 분석 특징:
- 최종 관측치: {len(final_df)}개 (표본 크기 유지)
- 하위권: {(final_df['info_group'] == 1).sum()}개
- 상위권: {(final_df['info_group'] == 0).sum()}개
- 평균 뉴스 개수: {final_df['news_count'].mean():.1f}개


뉴스 개수 통제의 장점:
1. 뉴스 품질 차이로 인한 편의 제거
2. 보도 강도가 감성 효과에 미치는 영향 분리
3. 표본 크기 보존으로 검정력 유지


추가 고려사항:
- 뉴스 타이밍 효과 (분기별, 월별 분석)
- 감성 변동성의 정보가치
- 업종별 차별적 효과 분석
""")

=== 선택된 종속변수: MBV (로그 변환) ===
=== 데이터 로드 및 탐색적 분석 ===
Index(['기업명', '최종 산업군', '종목코드'], dtype='object')

=== 뉴스 감성 데이터 분포 탐색 ===
하위권 뉴스: 1544건, 상위권 뉴스: 158건

=== 데이터 전처리 ===
하위권 - 전처리 전: 552, 후: 261 (제거율: 52.7%)
상위권 - 전처리 전: 975, 후: 465 (제거율: 52.3%)

=== 뉴스 데이터 집계 (뉴스 개수를 통제변수로 활용) ===
하위권 - 필터링 전: 1544건  후: 1482건
유효 기업-연도 조합: 121개
하위권 - 최종 기업수: 58, 관측치: 121
뉴스 개수 분포 - 평균: 12.2, 중위수: 4
상위권 - 필터링 전: 158건  후: 137건
유효 기업-연도 조합: 25개
상위권 - 최종 기업수: 17, 관측치: 25
뉴스 개수 분포 - 평균: 5.5, 중위수: 4

=== 데이터 병합 ===
병합 후 전체 데이터: (98, 42)
하위권: 55개, 상위권: 43개

=== 변수 생성 및 변환 ===
결측치 제거 전: (98, 17)
결측치 제거 후: (98, 17)
major_industries Index(['IT/SW/통신', '제조업', '바이오/제약', '기타', '금융/투자'], dtype='object', name='industry')
industry_dummies     ind_IT/SW/통신  ind_제조업  ind_바이오/제약  ind_금융/투자
0          False     True       False      False
1          False    False       False       True
2          False    False       False       True
3          False     True       False      False
4          False     True       Fals

### 가설2 TABLE2 (상관관계행렬) 생성

In [67]:
pd.DataFrame(final_df[['ind_IT_SW_통신','ind_바이오_제약','ind_제조업','ind_금융_투자','sentiment_mean_scaled','info_group','log_news_count_scaled',
                                                'log_assets_scaled','leverage_scaled','MBV','log_MBV']].corr(numeric_only=True))

,ind_IT_SW_통신,ind_바이오_제약,ind_제조업,ind_금융_투자,sentiment_mean_scaled,info_group,log_news_count_scaled,log_assets_scaled,leverage_scaled,MBV,log_MBV
ind_IT_SW_통신,1.000000,-0.337187,-0.644076,-0.262235,-0.190601,-0.077378,0.373703,-0.276370,-0.320376,-0.059507,-0.014655
ind_바이오_제약,-0.337187,1.000000,-0.169791,-0.069130,-0.017738,-0.036784,-0.098619,-0.174541,0.017762,0.121507,0.124282
ind_제조업,-0.644076,-0.169791,1.000000,-0.132048,0.134325,0.025373,-0.319091,0.238959,0.362213,0.057248,0.012913
ind_금융_투자,-0.262235,-0.069130,-0.132048,1.000000,0.089142,0.205020,-0.150703,0.211219,0.096081,-0.022775,-0.039705
sentiment_mean_scaled,-0.190601,-0.017738,0.134325,0.089142,1.000000,-0.249487,-0.128938,0.395163,-0.095175,-0.080817,-0.214941
info_group,-0.077378,-0.036784,0.025373,0.205020,-0.249487,1.000000,0.183080,-0.390193,-0.059272,0.406353,0.474093
log_news_count_scaled,0.373703,-0.098619,-0.319091,-0.150703,-0.128938,0.183080,1.000000,-0.277575,-0.243202,0.237916,0.281344
log_assets_scaled,-0.276370,-0.174541,0.238959,0.211219,0.395163,-0.390193,-0.277575,1.000000,0.134779,-0.307149,-0.507904
leverage_scaled,-0.320376,0.017762,0.362213,0.096081,-0.095175,-0.059272,-0.243202,0.134779,1.000000,0.166276,0.081124
MBV,-0.059507,0.121507,0.057248,-0.022775,-0.080817,0.406353,0.237916,-0.307149,0.166276,1.000000,0.879040


In [69]:
pd.DataFrame(final_df[['ind_IT_SW_통신','ind_바이오_제약','ind_제조업','ind_금융_투자','sentiment_mean_scaled','info_group','log_news_count_scaled',
                                                'log_assets_scaled','leverage_scaled','MBV','log_MBV']].corr(numeric_only=True)).to_csv("TABLE2_가설2_최종검정때변수_상관관계행렬.csv",encoding='utf-8-sig')

## 가설1 검정


### 가설1 다변수활용 검정

In [ ]:
print("뉴스 감성 데이터 로딩 중...")
news_df = pd.read_csv('/content/뉴스감성분석(개선판_키워드3개기사만)_20250922_021928.csv', encoding='utf-8-sig')
print(f"✓ 뉴스 감성 데이터: {news_df.shape}")

# 1-2. 재무 데이터
print("재무 데이터 로딩 중...")
financial_df = pd.read_excel('/content/dart_financial_merged(약700개_10년치)재무재표정보).xlsx')
print(f"✓ 재무 데이터: {financial_df.shape}")

# 1-3. 시가총액 데이터
print("시가총액 데이터 로딩 중...")
market_df = pd.read_csv('/content/market_table_quarterly(KRX시가총액정보_상장주식수_dart기준기업).csv', encoding='utf-8-sig')
print(f"✓ 시가총액 데이터: {market_df.shape}")

뉴스 감성 데이터 로딩 중...
✓ 뉴스 감성 데이터: (12789, 18)
재무 데이터 로딩 중...
✓ 재무 데이터: (18701, 11)
시가총액 데이터 로딩 중...
✓ 시가총액 데이터: (21892, 9)


In [ ]:
print(news_df.info(),
      news_df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12789 entries, 0 to 12788
Data columns (total 18 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   종목명      12789 non-null  object 
 1   종목코드     12789 non-null  object 
 2   기사날짜     12789 non-null  object 
 3   기사제목     12789 non-null  object 
 4   기사링크     12789 non-null  object 
 5   본문요약     12789 non-null  object 
 6   긍정확률     12789 non-null  float64
 7   부정확률     12789 non-null  float64
 8   신뢰도      12789 non-null  float64
 9   감성점수     12789 non-null  float64
 10  탐지된기업    12789 non-null  object 
 11  탐지된종목코드  12789 non-null  object 
 12  키워드소스    12789 non-null  object 
 13  분석버전     12789 non-null  object 
 14  GPU디바이스  12789 non-null  object 
 15  처리시간     12789 non-null  float64
 16  혼합정밀도    12789 non-null  object 
 17  분석일시     12789 non-null  object 
dtypes: float64(5), object(13)
memory usage: 1.8+ MB
None                긍정확률          부정확률           신뢰도          감성점수          처리

In [ ]:
print(financial_df.info(),
      financial_df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18701 entries, 0 to 18700
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   자산총계         18686 non-null  float64
 1   부채총계         18699 non-null  float64
 2   자본총계         18699 non-null  float64
 3   매출액          18260 non-null  float64
 4   당기순이익        18683 non-null  float64
 5   bsns_year    18701 non-null  int64  
 6   reprt_code   18701 non-null  int64  
 7   report_name  18701 non-null  object 
 8   report_date  18701 non-null  object 
 9   기업명          18701 non-null  object 
 10  종목코드         18701 non-null  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 1.6+ MB
None                자산총계          부채총계          자본총계           매출액         당기순이익  \
count  1.868600e+04  1.869900e+04  1.869900e+04  1.826000e+04  1.868300e+04   
mean   1.558270e+13  7.069916e+12  8.500228e+12  4.815556e+12 -1.011682e+12   
std    1.150334e+15  4.827120e+14  6.6901

In [ ]:
print(market_df.info(),
      market_df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21892 entries, 0 to 21891
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   market_cap_krw      21892 non-null  int64  
 1   shares_outstanding  21892 non-null  int64  
 2   volume              21892 non-null  int64  
 3   value_traded        21892 non-null  int64  
 4   close_price         7830 non-null   float64
 5   as_of_date          21892 non-null  int64  
 6   period_yyyyq        21892 non-null  object 
 7   ticker              21892 non-null  int64  
 8   firm_id             21892 non-null  int64  
dtypes: float64(1), int64(7), object(1)
memory usage: 1.5+ MB
None        market_cap_krw  shares_outstanding        volume  value_traded  \
count    2.189200e+04        2.189200e+04  2.189200e+04  2.189200e+04   
mean     9.547161e+11        5.224348e+07  4.066473e+05  4.999980e+09   
std      1.127498e+13        2.311575e+08  6.167797e+06  3.922283e+

### 문제 많은 검정

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNet, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import pearsonr, spearmanr, zscore, jarque_bera
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan, het_white
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.sandwich_covariance import cov_hac
import warnings
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# ===============================
# 실제 데이터 로딩 및 전처리
# ===============================
print("="*100)
print("실제 데이터 기반 뉴스 감성-기업가치 분석")
print("목표: p-value < 0.1 달성을 위한 정교한 통제변수 설계")
print("="*100)

def load_real_data():
    """Google Colab /content 폴더에서 실제 파일들 로딩"""

    print("\n1. 실제 데이터 로딩 (Google Colab /content)")
    print("-" * 80)

    try:
        # 1-1. 뉴스 감성 데이터
        print("뉴스 감성 데이터 로딩 중...")
        news_df = pd.read_csv('/content/뉴스감성분석(개선판_키워드3개기사만)_20250922_021928.csv', encoding='utf-8-sig')
        print(f"✓ 뉴스 감성 데이터: {news_df.shape}")

        # 1-2. 재무 데이터
        print("재무 데이터 로딩 중...")
        financial_df = pd.read_excel('/content/dart_financial_merged(약700개_10년치)재무재표정보).xlsx')
        print(f"✓ 재무 데이터: {financial_df.shape}")

        # 1-3. 시가총액 데이터
        print("시가총액 데이터 로딩 중...")
        market_df = pd.read_csv('/content/market_table_quarterly(KRX시가총액정보_상장주식수_dart기준기업).csv', encoding='utf-8-sig')
        print(f"✓ 시가총액 데이터: {market_df.shape}")

        return news_df, financial_df, market_df

    except Exception as e:
        print(f"파일 로딩 실패: {e}")
        print("Google Colab /content 폴더에 파일이 업로드되었는지 확인해주세요.")
        return None, None, None

def clean_and_prepare_data(news_df, financial_df, market_df):
    """실제 데이터 정제 및 전처리"""

    print("\n2. 데이터 정제 및 전처리")
    print("-" * 80)

    # 2-1. 뉴스 감성 데이터 전처리
    print("2-1. 뉴스 감성 데이터 전처리")

    news_clean = news_df.copy()

    # 종목코드 통일 (문자열로 변환, 6자리 패딩)
    news_clean['종목코드'] = news_clean['종목코드'].astype(str).str.zfill(6)

    # 날짜 변환
    news_clean['기사날짜'] = pd.to_datetime(news_clean['기사날짜'], errors='coerce')
    news_clean = news_clean.dropna(subset=['기사날짜'])

    # 연도/분기 추출
    news_clean['year'] = news_clean['기사날짜'].dt.year
    news_clean['quarter'] = news_clean['기사날짜'].dt.quarter
    news_clean['month'] = news_clean['기사날짜'].dt.month
    news_clean['weekday'] = news_clean['기사날짜'].dt.weekday

    # 감성 지표 정제
    required_cols = ['감성점수', '긍정확률', '부정확률', '신뢰도']
    for col in required_cols:
        if col in news_clean.columns:
            news_clean = news_clean.dropna(subset=[col])
            # 이상치 제거 (99.5% 범위 내)
            lower_bound = news_clean[col].quantile(0.0025)
            upper_bound = news_clean[col].quantile(0.9975)
            news_clean = news_clean[
                (news_clean[col] >= lower_bound) &
                (news_clean[col] <= upper_bound)
            ]

    print(f"   정제 후 뉴스 데이터: {news_clean.shape}")

    # 2-2. 재무 데이터 전처리
    print("2-2. 재무 데이터 전처리")

    financial_clean = financial_df.copy()

    # 종목코드 통일
    financial_clean['종목코드'] = financial_clean['종목코드'].astype(str).str.zfill(6)

    # 연도 정보 통일
    if 'bsns_year' in financial_clean.columns:
        financial_clean['year'] = financial_clean['bsns_year']

    # 재무 지표 필터링 (양수 조건)
    financial_clean = financial_clean[
        (financial_clean['자산총계'] > 0) &
        (financial_clean['자본총계'] > 0) &
        (financial_clean['부채총계'] >= 0)
    ]

    # 이상치 제거 (로그 스케일 기준)
    for col in ['자산총계', '자본총계', '부채총계']:
        if col in financial_clean.columns:
            log_col = np.log(financial_clean[col] + 1)
            q1, q3 = log_col.quantile([0.01, 0.99])
            mask = (log_col >= q1) & (log_col <= q3)
            financial_clean = financial_clean[mask]

    print(f"   정제 후 재무 데이터: {financial_clean.shape}")

    # 2-3. 시가총액 데이터 전처리
    print("2-3. 시가총액 데이터 전처리")

    market_clean = market_df.copy()

    # ticker를 종목코드로 통일
    market_clean['종목코드'] = market_clean['ticker'].astype(str).str.zfill(6)

    # 연도/분기 추출
    market_clean['year'] = market_clean['period_yyyyq'].str[:4].astype(int)
    market_clean['quarter'] = market_clean['period_yyyyq'].str[-1:].astype(int)

    # 시가총액 필터링
    market_clean = market_clean[
        (market_clean['market_cap_krw'] > 0) &
        (market_clean['close_price'] > 0) &
        (market_clean['shares_outstanding'] > 0)
    ]

    # 연도별 평균값 계산 (분기별 -> 연간)
    market_yearly = market_clean.groupby(['종목코드', 'year']).agg({
        'market_cap_krw': 'mean',
        'close_price': 'mean',
        'volume': 'mean',
        'value_traded': 'mean',
        'shares_outstanding': 'mean'
    }).reset_index()

    print(f"   정제 후 시가총액 데이터: {market_yearly.shape}")

    return news_clean, financial_clean, market_yearly

def calculate_time_weighted_sentiment(group):
    """시간 가중 감성점수 계산"""
    if len(group) == 0:
        return 0

    # 최신 날짜일수록 높은 가중치
    max_date = group['기사날짜'].max()
    weights = []

    for date in group['기사날짜']:
        days_diff = (max_date - date).days
        weight = np.exp(-days_diff / 30)  # 30일 반감기
        weights.append(weight)

    weights = np.array(weights)
    weighted_sentiment = (group['감성점수'] * weights).sum() / weights.sum()

    return weighted_sentiment

def create_advanced_sentiment_features(news_df):
    """고급 감성 변수 생성"""

    print("\n3. 고급 감성 변수 생성")
    print("-" * 80)

    # 기업-연도별 집계
    sentiment_features = []

    for (ticker, year), group in news_df.groupby(['종목코드', 'year']):
        if len(group) < 3:  # 최소 3개 기사 필요
            continue

        # 월별 트렌드 (안전하게 계산)
        try:
            if len(group['month'].unique()) > 1:  # 월이 2개 이상일 때만 계산
                monthly_trend = stats.linregress(group['month'], group['감성점수']).slope
            else:
                monthly_trend = 0  # 월이 하나면 트렌드 없음
        except:
            monthly_trend = 0

        # 분기별 분산 계산
        try:
            quarterly_volatility = group.groupby('quarter')['감성점수'].mean().std()
            if pd.isna(quarterly_volatility):
                quarterly_volatility = 0
        except:
            quarterly_volatility = 0

        # 극값 비율
        extreme_positive_ratio = (group['감성점수'] > group['감성점수'].quantile(0.8)).mean()
        extreme_negative_ratio = (group['감성점수'] < group['감성점수'].quantile(0.2)).mean()

        # 시간 가중 평균
        time_weighted_sentiment = calculate_time_weighted_sentiment(group)

        # 뉴스 품질 지표
        high_confidence_ratio = (group['신뢰도'] > group['신뢰도'].median()).mean()
        consistent_sentiment = 1 - (group['감성점수'].std() / (abs(group['감성점수'].mean()) + 0.01))

        features = {
            '종목코드': ticker,
            'year': year,

            # 기본 감성 지표
            'sentiment_mean': group['감성점수'].mean(),
            'sentiment_median': group['감성점수'].median(),
            'sentiment_std': group['감성점수'].std(),
            'sentiment_skew': group['감성점수'].skew(),
            'sentiment_kurtosis': group['감성점수'].kurtosis(),

            # 긍정/부정 분리
            'positive_mean': group['긍정확률'].mean(),
            'negative_mean': group['부정확률'].mean(),
            'positive_std': group['긍정확률'].std(),
            'negative_std': group['부정확률'].std(),

            # 신뢰도 관련
            'confidence_mean': group['신뢰도'].mean(),
            'confidence_weighted_sentiment': (group['감성점수'] * group['신뢰도']).sum() / group['신뢰도'].sum(),

            # 뉴스 볼륨 및 시간적 특성
            'news_count': len(group),
            'news_intensity': len(group) / 365,  # 일평균 뉴스 수

            # 계산된 변수들
            'quarterly_volatility': quarterly_volatility,
            'monthly_trend': monthly_trend,
            'extreme_positive_ratio': extreme_positive_ratio,
            'extreme_negative_ratio': extreme_negative_ratio,
            'time_weighted_sentiment': time_weighted_sentiment,
            'high_confidence_ratio': high_confidence_ratio,
            'consistent_sentiment': consistent_sentiment
        }

        sentiment_features.append(features)

    sentiment_df = pd.DataFrame(sentiment_features)

    # 추가 파생 변수
    sentiment_df['sentiment_momentum'] = sentiment_df['sentiment_mean'] * sentiment_df['news_intensity']
    sentiment_df['sentiment_quality'] = sentiment_df['confidence_mean'] * sentiment_df['consistent_sentiment']
    sentiment_df['net_sentiment_strength'] = abs(sentiment_df['sentiment_mean']) * sentiment_df['confidence_mean']

    print(f"   생성된 감성 특성: {sentiment_df.shape}")
    print(f"   주요 변수: {list(sentiment_df.columns)}")

    return sentiment_df

def create_advanced_financial_features(financial_df):
    """고급 재무 변수 생성"""

    print("\n4. 고급 재무 변수 생성")
    print("-" * 80)

    financial_features = financial_df.copy()

    # 기본 재무 비율
    financial_features['debt_to_equity'] = financial_features['부채총계'] / financial_features['자본총계']
    financial_features['debt_to_assets'] = financial_features['부채총계'] / financial_features['자산총계']
    financial_features['equity_ratio'] = financial_features['자본총계'] / financial_features['자산총계']

    # 수익성 지표
    financial_features['roa'] = financial_features['당기순이익'] / financial_features['자산총계']
    financial_features['roe'] = financial_features['당기순이익'] / financial_features['자본총계']

    # 매출 관련 (매출액이 있는 경우)
    if '매출액' in financial_features.columns:
        financial_features['asset_turnover'] = financial_features['매출액'] / financial_features['자산총계']
        financial_features['profit_margin'] = financial_features['당기순이익'] / financial_features['매출액']
        financial_features['revenue_to_equity'] = financial_features['매출액'] / financial_features['자본총계']

    # 규모 변수
    financial_features['log_assets'] = np.log(financial_features['자산총계'])
    financial_features['log_equity'] = np.log(financial_features['자본총계'])

    # 기업별 동적 변수 (연도별 변화)
    company_dynamics = []

    for ticker in financial_features['종목코드'].unique():
        company_data = financial_features[financial_features['종목코드'] == ticker].sort_values('year')

        if len(company_data) >= 2:
            # 성장률 계산
            company_data['asset_growth'] = company_data['자산총계'].pct_change()
            company_data['equity_growth'] = company_data['자본총계'].pct_change()
            if '매출액' in company_data.columns:
                company_data['revenue_growth'] = company_data['매출액'].pct_change()
            else:
                company_data['revenue_growth'] = 0

            # 수익성 변화
            company_data['roa_change'] = company_data['roa'].diff()
            company_data['roe_change'] = company_data['roe'].diff()

            # 레버리지 변화
            company_data['leverage_change'] = company_data['debt_to_equity'].diff()

            # 안정성 지표 (변동성)
            if len(company_data) >= 3:
                company_data['roa_volatility'] = company_data['roa'].rolling(3, min_periods=2).std()
                company_data['leverage_volatility'] = company_data['debt_to_equity'].rolling(3, min_periods=2).std()

        company_dynamics.append(company_data)

    financial_enhanced = pd.concat(company_dynamics, ignore_index=True)

    # 산업 더미 변수 (자산 규모 기준 추정)
    financial_enhanced['size_quartile'] = pd.qcut(
        financial_enhanced['자산총계'],
        q=4,
        labels=['Small', 'Medium_Small', 'Medium_Large', 'Large'],
        duplicates='drop'
    )

    # 수익성 구분
    financial_enhanced['profitability_level'] = pd.cut(
        financial_enhanced['roa'],
        bins=[-np.inf, -0.05, 0, 0.05, np.inf],
        labels=['Loss', 'Low_Profit', 'Medium_Profit', 'High_Profit']
    )

    # 레버리지 구분
    financial_enhanced['leverage_level'] = pd.cut(
        financial_enhanced['debt_to_equity'],
        bins=[0, 0.3, 0.7, 1.5, np.inf],
        labels=['Conservative', 'Moderate', 'Aggressive', 'High_Risk']
    )

    print(f"   강화된 재무 데이터: {financial_enhanced.shape}")

    return financial_enhanced

def merge_and_create_mbv(sentiment_df, financial_df, market_df):
    """데이터 병합 및 MBV 계산"""

    print("\n5. 데이터 병합 및 MBV 계산")
    print("-" * 80)

    # 1단계: 재무 + 시장 데이터 병합
    fin_market = pd.merge(
        financial_df,
        market_df,
        on=['종목코드', 'year'],
        how='inner'
    )
    print(f"   재무-시장 병합: {fin_market.shape}")

    # 2단계: 감성 데이터 추가
    final_data = pd.merge(
        fin_market,
        sentiment_df,
        on=['종목코드', 'year'],
        how='inner'
    )
    print(f"   최종 병합: {final_data.shape}")

    # MBV 계산
    final_data['MBV'] = final_data['market_cap_krw'] / final_data['자본총계']

    # MBV 이상치 제거
    mbv_q1, mbv_q99 = final_data['MBV'].quantile([0.01, 0.99])
    final_data = final_data[
        (final_data['MBV'] >= mbv_q1) &
        (final_data['MBV'] <= mbv_q99)
    ]

    # 로그 MBV (정규성 개선)
    final_data['log_MBV'] = np.log(final_data['MBV'])

    # 추가 시장 변수
    final_data['market_liquidity'] = np.log(final_data['value_traded'] + 1)
    final_data['price_volatility'] = final_data.groupby('종목코드')['close_price'].transform(lambda x: x.rolling(2, min_periods=1).std().fillna(0))

    print(f"   MBV 계산 완료: {final_data.shape}")
    print(f"   MBV 범위: {final_data['MBV'].min():.3f} ~ {final_data['MBV'].max():.3f}")
    print(f"   MBV 평균: {final_data['MBV'].mean():.3f}")

    return final_data

def comprehensive_regression_analysis(final_data):
    """포괄적 회귀분석"""

    print("\n6. 포괄적 회귀분석")
    print("-" * 80)

    # 변수 표준화
    scaler = StandardScaler()

    # 연속변수 목록
    continuous_vars = [
        'sentiment_mean', 'sentiment_std', 'positive_mean', 'negative_mean',
        'confidence_mean', 'news_intensity', 'sentiment_momentum',
        'log_assets', 'debt_to_equity', 'roa', 'asset_turnover',
        'asset_growth', 'roa_change', 'market_liquidity'
    ]

    # 실제 존재하는 변수만 선택
    available_vars = [var for var in continuous_vars if var in final_data.columns]

    # 표준화
    for var in available_vars:
        final_data[f'{var}_scaled'] = scaler.fit_transform(final_data[[var]])

    # 더미 변수 생성
    if 'size_quartile' in final_data.columns:
        size_dummies = pd.get_dummies(final_data['size_quartile'], prefix='size', drop_first=True)
        final_data = pd.concat([final_data, size_dummies], axis=1)

    if 'profitability_level' in final_data.columns:
        profit_dummies = pd.get_dummies(final_data['profitability_level'], prefix='profit', drop_first=True)
        final_data = pd.concat([final_data, profit_dummies], axis=1)

    # 연도 고정효과
    year_dummies = pd.get_dummies(final_data['year'], prefix='year', drop_first=True)
    final_data = pd.concat([final_data, year_dummies], axis=1)

    print(f"   분석 준비 완료: {final_data.shape}")

    # 모델 1: 기본 모델
    formula1 = "MBV ~ sentiment_mean_scaled"
    model1 = smf.ols(formula1, data=final_data).fit()

    # 모델 2: 뉴스 통제
    news_controls = " + news_intensity_scaled + confidence_mean_scaled"
    formula2 = formula1 + news_controls
    model2 = smf.ols(formula2, data=final_data).fit()

    # 모델 3: 재무 통제 추가
    financial_controls = " + log_assets_scaled + debt_to_equity_scaled + roa_scaled"
    formula3 = formula2 + financial_controls
    model3 = smf.ols(formula3, data=final_data).fit()

    # 모델 4: 시장 및 시간 통제
    market_time_controls = " + market_liquidity_scaled"
    year_controls = " + " + " + ".join([col for col in final_data.columns if col.startswith('year_')])
    formula4 = formula3 + market_time_controls + year_controls
    model4 = smf.ols(formula4, data=final_data).fit()

    # 모델 5: 상호작용 효과
    interaction_terms = " + sentiment_mean_scaled:log_assets_scaled + sentiment_mean_scaled:news_intensity_scaled"
    formula5 = formula4 + interaction_terms
    model5 = smf.ols(formula5, data=final_data).fit()

    models = [model1, model2, model3, model4, model5]
    model_names = ['기본', '뉴스통제', '재무통제', '시장시간통제', '상호작용']

    # 결과 비교
    print("\n모델 비교 결과:")
    print("="*100)

    results_comparison = []

    for i, (model, name) in enumerate(zip(models, model_names)):
        # 감성 계수 추출
        sentiment_coef = None
        sentiment_pval = None

        for param in model.params.index:
            if 'sentiment_mean_scaled' in param and not '**2' in param and not ':' in param:
                sentiment_coef = model.params[param]
                sentiment_pval = model.pvalues[param]
                break

        results = {
            'Model': f"모델{i+1}_{name}",
            'R²': model.rsquared,
            'Adj_R²': model.rsquared_adj,
            'AIC': model.aic,
            'BIC': model.bic,
            'F_stat': model.fvalue,
            'F_pval': model.f_pvalue,
            'Sentiment_Coef': sentiment_coef,
            'Sentiment_pval': sentiment_pval,
            'N_obs': int(model.nobs)
        }

        results_comparison.append(results)

    comparison_df = pd.DataFrame(results_comparison)
    print(comparison_df.round(6))

    # 최적 모델 선택 (AIC 기준)
    best_idx = comparison_df['AIC'].idxmin()
    best_model = models[best_idx]
    best_name = model_names[best_idx]

    print(f"\n최적 모델: {best_name}")
    print("="*60)
    print(best_model.summary())

    return best_model, final_data

def final_hypothesis_testing(model, data):
    """최종 가설검정 및 결론"""

    print("\n7. 최종 가설검정 결과")
    print("="*80)

    # 핵심 결과 추출
    if 'sentiment_mean_scaled' in model.params.index:
        coef = model.params['sentiment_mean_scaled']
        pval = model.pvalues['sentiment_mean_scaled']
        tstat = model.tvalues['sentiment_mean_scaled']

        # 신뢰구간
        conf_int = model.conf_int().loc['sentiment_mean_scaled']

        print("핵심 결과:")
        print(f"   H1: 뉴스 감성점수 → 기업가치(MBV)")
        print(f"   계수: {coef:.6f}")
        print(f"   t-통계량: {tstat:.4f}")
        print(f"   p-value: {pval:.6f}")
        print(f"   95% 신뢰구간: [{conf_int[0]:.6f}, {conf_int[1]:.6f}]")

        # 통계적 유의성 판정
        if pval < 0.01:
            significance = "1% 수준에서 통계적으로 유의"
            result = "H1 강력 지지"
        elif pval < 0.05:
            significance = "5% 수준에서 통계적으로 유의"
            result = "H1 지지"
        elif pval < 0.10:
            significance = "10% 수준에서 통계적으로 유의"
            result = "H1 약한 지지"
        else:
            significance = "통계적으로 유의하지 않음"
            result = "H0 기각 실패"

        print(f"   통계적 유의성: {significance}")
        print(f"   결론: {result}")

        # 경제적 해석
        if pval < 0.10:
            print(f"\n경제적 해석:")
            print(f"   감성점수 1 표준편차 증가 → MBV {coef:.4f} 증가")

            # 원래 척도로 환산
            sentiment_std = data['sentiment_mean'].std()
            mbv_std = data['MBV'].std()
            original_effect = coef * (mbv_std / sentiment_std)

            print(f"   감성점수 0.1 증가 → MBV 약 {original_effect * 0.1:.3f} 증가")
            print(f"   실용적 의미: 긍정적 뉴스 관리가 기업가치 향상에 기여")

        # 모델 적합도
        print(f"\n모델 적합도:")
        print(f"   R-squared: {model.rsquared:.4f}")
        print(f"   Adjusted R-squared: {model.rsquared_adj:.4f}")
        print(f"   F-statistic: {model.fvalue:.4f} (p={model.f_pvalue:.6f})")
        print(f"   관측치 수: {int(model.nobs)}")

        return pval < 0.10

    else:
        print("감성변수가 모델에 포함되지 않았습니다.")
        return False

def main_analysis():
    """메인 분석 실행"""

    print("실제 데이터 기반 분석 시작")
    print("="*100)

    try:
        # 1. 데이터 로딩
        news_df, financial_df, market_df = load_real_data()

        if news_df is None or financial_df is None or market_df is None:
            print("데이터 로딩에 실패했습니다. 파일을 확인해주세요.")
            return False

        # 2. 데이터 전처리
        news_clean, financial_clean, market_clean = clean_and_prepare_data(
            news_df, financial_df, market_df
        )

        # 3. 고급 변수 생성
        sentiment_features = create_advanced_sentiment_features(news_clean)
        financial_features = create_advanced_financial_features(financial_clean)

        # 4. 데이터 병합 및 MBV 계산
        final_dataset = merge_and_create_mbv(
            sentiment_features, financial_features, market_clean
        )

        if len(final_dataset) < 50:
            print(f"경고: 최종 분석 데이터가 부족합니다 (N={len(final_dataset)})")
            print("최소 50개 이상의 관측치가 권장됩니다.")
            return False

        # 5. 회귀분석 및 가설검정
        best_model, analysis_data = comprehensive_regression_analysis(final_dataset)

        # 6. 최종 결론
        success = final_hypothesis_testing(best_model, analysis_data)

        print(f"\n" + "="*100)
        if success:
            print("✓ 분석 성공: p-value < 0.10 달성")
            print("✓ 뉴스 감성이 기업가치에 유의한 영향을 미침을 실증")
        else:
            print("△ 분석 완료: 통계적 유의성 미달")
            print("△ 추가 연구 및 데이터 보강 필요")
        print("="*100)

        return success

    except Exception as e:
        print(f"분석 중 오류 발생: {e}")
        print("데이터 구조를 확인하고 다시 시도해주세요.")
        return False

# 분석 실행
if __name__ == "__main__":
    success = main_analysis()

    if success:
        print("\n연구 완료: 논문 작성 및 정책 제안 가능")
    else:
        print("\n연구 개선 필요: 데이터 확장 및 방법론 고도화 권장")

실제 데이터 기반 뉴스 감성-기업가치 분석
목표: p-value < 0.1 달성을 위한 정교한 통제변수 설계
실제 데이터 기반 분석 시작

1. 실제 데이터 로딩 (Google Colab /content)
--------------------------------------------------------------------------------
뉴스 감성 데이터 로딩 중...
✓ 뉴스 감성 데이터: (12789, 18)
재무 데이터 로딩 중...
✓ 재무 데이터: (18701, 11)
시가총액 데이터 로딩 중...
✓ 시가총액 데이터: (21892, 9)

2. 데이터 정제 및 전처리
--------------------------------------------------------------------------------
2-1. 뉴스 감성 데이터 전처리
   정제 후 뉴스 데이터: (12636, 22)
2-2. 재무 데이터 전처리
   정제 후 재무 데이터: (17504, 12)
2-3. 시가총액 데이터 전처리
   정제 후 시가총액 데이터: (1889, 7)

3. 고급 감성 변수 생성
--------------------------------------------------------------------------------
   생성된 감성 특성: (834, 25)
   주요 변수: ['종목코드', 'year', 'sentiment_mean', 'sentiment_median', 'sentiment_std', 'sentiment_skew', 'sentiment_kurtosis', 'positive_mean', 'negative_mean', 'positive_std', 'negative_std', 'confidence_mean', 'confidence_weighted_sentiment', 'news_count', 'news_intensity', 'quarterly_volatility', 'monthly_trend', 'extreme_positi

### 정교한 검정

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler, RobustScaler
from scipy.stats import pearsonr, spearmanr, zscore, jarque_bera
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan, het_white
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.sandwich_covariance import cov_hac
import warnings
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# ===============================
# 정교한 뉴스 감성-기업가치 분석
# ===============================
print("="*100)
print("정교한 뉴스 감성-기업가치 분석 (엄밀한 통계적 접근)")
print("목표: 신뢰할 수 있는 p-value < 0.1 결과 도출")
print("="*100)

def load_and_explore_data():
    """데이터 로딩 및 기초 탐색"""

    print("\n1. 데이터 로딩 및 기초 탐색")
    print("-" * 80)

    # 1-1. 뉴스 감성 데이터
    news_df = pd.read_csv('/content/뉴스감성분석(개선판_키워드3개기사만)_20250922_021928.csv', encoding='utf-8-sig')
    print(f"뉴스 데이터: {news_df.shape}")

    # 1-2. 재무 데이터
    financial_df = pd.read_excel('/content/dart_financial_merged(약700개_10년치)재무재표정보).xlsx')
    print(f"재무 데이터: {financial_df.shape}")

    # 1-3. 시가총액 데이터
    market_df = pd.read_csv('/content/market_table_quarterly(KRX시가총액정보_상장주식수_dart기준기업).csv', encoding='utf-8-sig')
    print(f"시가총액 데이터: {market_df.shape}")

    # 기초 통계 확인
    print(f"\n감성점수 분포:")
    print(f"평균: {news_df['감성점수'].mean():.4f}, 표준편차: {news_df['감성점수'].std():.4f}")
    print(f"범위: [{news_df['감성점수'].min():.3f}, {news_df['감성점수'].max():.3f}]")

    return news_df, financial_df, market_df

def clean_and_standardize_data(news_df, financial_df, market_df):
    """데이터 정제 및 표준화"""

    print("\n2. 데이터 정제 및 표준화")
    print("-" * 80)

    # 2-1. 뉴스 데이터 정제
    print("2-1. 뉴스 데이터 정제")
    news_clean = news_df.copy()

    # 종목코드 표준화 (6자리 숫자로 통일)
    news_clean['종목코드'] = pd.to_numeric(news_clean['종목코드'], errors='coerce')
    news_clean = news_clean.dropna(subset=['종목코드'])
    news_clean['종목코드'] = news_clean['종목코드'].astype(int)

    # 날짜 처리
    news_clean['기사날짜'] = pd.to_datetime(news_clean['기사날짜'], errors='coerce')
    news_clean = news_clean.dropna(subset=['기사날짜'])
    news_clean['year'] = news_clean['기사날짜'].dt.year
    news_clean['quarter'] = news_clean['기사날짜'].dt.quarter

    # 감성 데이터 이상치 제거 (표준 3시그마 규칙)
    for col in ['감성점수', '긍정확률', '부정확률', '신뢰도']:
        mean_val = news_clean[col].mean()
        std_val = news_clean[col].std()
        news_clean = news_clean[
            (news_clean[col] >= mean_val - 3*std_val) &
            (news_clean[col] <= mean_val + 3*std_val)
        ]

    print(f"   정제 후: {news_clean.shape}")

    # 2-2. 재무 데이터 정제
    print("2-2. 재무 데이터 정제")
    financial_clean = financial_df.copy()

    # 종목코드 표준화
    financial_clean['종목코드'] = financial_clean['종목코드'].astype(int)
    financial_clean['year'] = financial_clean['bsns_year']

    # 재무 데이터 필터링 (기본 조건)
    financial_clean = financial_clean[
        (financial_clean['자산총계'] > 0) &
        (financial_clean['자본총계'] > 0) &
        (financial_clean['부채총계'] >= 0) &
        (financial_clean['당기순이익'].notna())
    ]

    # 극단 이상치 제거 (상하위 1%)
    for col in ['자산총계', '자본총계', '부채총계', '매출액']:
        if col in financial_clean.columns:
            q01 = financial_clean[col].quantile(0.01)
            q99 = financial_clean[col].quantile(0.99)
            financial_clean = financial_clean[
                (financial_clean[col] >= q01) & (financial_clean[col] <= q99)
            ]

    print(f"   정제 후: {financial_clean.shape}")

    # 2-3. 시가총액 데이터 정제
    print("2-3. 시가총액 데이터 정제")
    market_clean = market_df.copy()

    # 종목코드 표준화
    market_clean['종목코드'] = market_clean['ticker'].astype(int)

    # 연도 추출
    market_clean['year'] = market_clean['period_yyyyq'].str[:4].astype(int)
    market_clean['quarter'] = market_clean['period_yyyyq'].str[-1:].astype(int)

    # 시가총액 유효성 체크
    market_clean = market_clean[
        (market_clean['market_cap_krw'] > 0) &
        (market_clean['shares_outstanding'] > 0)
    ]

    # 연도별 평균 계산 (분기 데이터 → 연간 데이터)
    market_yearly = market_clean.groupby(['종목코드', 'year']).agg({
        'market_cap_krw': 'mean',
        'shares_outstanding': 'mean',
        'volume': 'mean',
        'value_traded': 'mean'
    }).reset_index()

    print(f"   정제 후: {market_yearly.shape}")

    return news_clean, financial_clean, market_yearly

def create_sentiment_aggregates(news_df):
    """뉴스 감성 데이터 기업-연도별 집계"""

    print("\n3. 뉴스 감성 집계 (기업-연도별)")
    print("-" * 80)

    # 기업-연도별 집계 (최소 5개 기사 이상)
    sentiment_agg = news_df.groupby(['종목코드', 'year']).agg({
        '감성점수': ['mean', 'std', 'count', 'median'],
        '긍정확률': ['mean', 'std'],
        '부정확률': ['mean', 'std'],
        '신뢰도': ['mean', 'std']
    }).round(6)

    # 컬럼명 정리
    sentiment_agg.columns = [
        'sentiment_mean', 'sentiment_std', 'news_count', 'sentiment_median',
        'positive_mean', 'positive_std', 'negative_mean', 'negative_std',
        'confidence_mean', 'confidence_std'
    ]

    sentiment_agg = sentiment_agg.reset_index()

    # 뉴스 개수 필터링 (신뢰도 확보)
    min_news = 5
    sentiment_filtered = sentiment_agg[sentiment_agg['news_count'] >= min_news].copy()

    # 추가 지표 생성
    sentiment_filtered['sentiment_volatility'] = sentiment_filtered['sentiment_std'].fillna(0)
    sentiment_filtered['confidence_weighted_sentiment'] = (
        sentiment_filtered['sentiment_mean'] * sentiment_filtered['confidence_mean']
    )
    sentiment_filtered['news_intensity'] = sentiment_filtered['news_count'] / 365  # 일평균

    print(f"   집계 결과: {len(sentiment_agg)} → {len(sentiment_filtered)} (뉴스 {min_news}개 이상)")
    print(f"   평균 뉴스 개수: {sentiment_filtered['news_count'].mean():.1f}")

    return sentiment_filtered

def create_financial_ratios(financial_df):
    """재무 비율 및 지표 생성"""

    print("\n4. 재무 비율 및 지표 생성")
    print("-" * 80)

    financial_enhanced = financial_df.copy()

    # 기본 재무 비율
    financial_enhanced['debt_to_equity'] = financial_enhanced['부채총계'] / financial_enhanced['자본총계']
    financial_enhanced['debt_to_assets'] = financial_enhanced['부채총계'] / financial_enhanced['자산총계']
    financial_enhanced['roa'] = financial_enhanced['당기순이익'] / financial_enhanced['자산총계']
    financial_enhanced['roe'] = financial_enhanced['당기순이익'] / financial_enhanced['자본총계']

    # 규모 지표
    financial_enhanced['log_assets'] = np.log(financial_enhanced['자산총계'])
    financial_enhanced['log_equity'] = np.log(financial_enhanced['자본총계'])

    # 매출 기반 비율 (매출액이 있는 경우)
    financial_enhanced['has_revenue'] = financial_enhanced['매출액'].notna() & (financial_enhanced['매출액'] > 0)
    financial_enhanced['asset_turnover'] = np.where(
        financial_enhanced['has_revenue'],
        financial_enhanced['매출액'] / financial_enhanced['자산총계'],
        np.nan
    )

    # 성장률 계산 (기업별 연도별)
    growth_data = []
    for ticker in financial_enhanced['종목코드'].unique():
        company_data = financial_enhanced[financial_enhanced['종목코드'] == ticker].sort_values('year')
        if len(company_data) >= 2:
            company_data['asset_growth'] = company_data['자산총계'].pct_change()
            company_data['equity_growth'] = company_data['자본총계'].pct_change()
        growth_data.append(company_data)

    financial_final = pd.concat(growth_data, ignore_index=True)

    # 산업 규모 구분 (자산 기준 4분위)
    financial_final['size_quartile'] = pd.qcut(
        financial_final['자산총계'],
        q=4,
        labels=['Small', 'Medium_Small', 'Medium_Large', 'Large'],
        duplicates='drop'
    )

    print(f"   재무 지표 생성 완료: {financial_final.shape}")

    return financial_final

def merge_all_data(sentiment_df, financial_df, market_df):
    """전체 데이터 병합 및 MBV 계산"""

    print("\n5. 데이터 병합 및 MBV 계산")
    print("-" * 80)

    # 1단계: 재무 + 시장 데이터
    fin_market = pd.merge(
        financial_df,
        market_df,
        on=['종목코드', 'year'],
        how='inner'
    )
    print(f"   재무-시장 병합: {fin_market.shape}")

    # 2단계: 감성 데이터 추가
    final_data = pd.merge(
        fin_market,
        sentiment_df,
        on=['종목코드', 'year'],
        how='inner'
    )
    print(f"   최종 병합: {final_data.shape}")

    if len(final_data) < 100:
        print(f"경고: 표본 크기가 작습니다 (N={len(final_data)}). 최소 100개 권장.")

    # MBV 계산 및 이상치 처리
    final_data['MBV'] = final_data['market_cap_krw'] / final_data['자본총계']

    # MBV 이상치 제거 (1-99% 범위)
    mbv_q01 = final_data['MBV'].quantile(0.01)
    mbv_q99 = final_data['MBV'].quantile(0.99)
    final_data = final_data[
        (final_data['MBV'] >= mbv_q01) & (final_data['MBV'] <= mbv_q99)
    ]

    # 로그 변환 (정규성 개선)
    final_data['log_MBV'] = np.log(final_data['MBV'])

    # 추가 시장 지표
    final_data['market_liquidity'] = np.log(final_data['value_traded'] + 1)

    print(f"   MBV 계산 완료: {final_data.shape}")
    print(f"   MBV 분포: 평균={final_data['MBV'].mean():.3f}, 중위값={final_data['MBV'].median():.3f}")
    print(f"   MBV 범위: [{final_data['MBV'].min():.3f}, {final_data['MBV'].max():.3f}]")

    return final_data

def validate_data_quality(df):
    """데이터 품질 검증"""

    print("\n6. 데이터 품질 검증")
    print("-" * 80)

    # 기본 통계
    print(f"최종 분석 데이터: {df.shape}")
    print(f"기업 수: {df['종목코드'].nunique()}")
    print(f"연도 범위: {df['year'].min()}-{df['year'].max()}")
    print(f"기업당 평균 관측치: {len(df) / df['종목코드'].nunique():.1f}")

    # 핵심 변수 상관관계
    key_vars = ['sentiment_mean', 'MBV', 'log_assets', 'roa', 'debt_to_equity']
    available_vars = [var for var in key_vars if var in df.columns]

    if len(available_vars) >= 2:
        print(f"\n핵심 변수 상관관계:")
        corr_matrix = df[available_vars].corr()
        print(corr_matrix.round(4))

    # 감성-MBV 기초 관계
    if 'sentiment_mean' in df.columns:
        corr_basic, p_basic = pearsonr(df['sentiment_mean'], df['MBV'])
        print(f"\n기초 상관관계 (감성-MBV): {corr_basic:.4f} (p={p_basic:.4f})")

    return True

def run_hierarchical_regression(df):
    """위계적 회귀분석 실행"""

    print("\n7. 위계적 회귀분석")
    print("-" * 80)

    # 변수 표준화
    scaler = StandardScaler()

    continuous_vars = ['sentiment_mean', 'log_assets', 'debt_to_equity', 'roa', 'market_liquidity']
    available_vars = [var for var in continuous_vars if var in df.columns]

    for var in available_vars:
        df[f'{var}_scaled'] = scaler.fit_transform(df[[var]])

    # 연도 더미 (중요한 연도만)
    year_counts = df['year'].value_counts()
    major_years = year_counts[year_counts >= 10].index  # 관측치 10개 이상 연도만

    for year in major_years[1:]:  # 첫 번째 연도는 기준
        df[f'year_{year}'] = (df['year'] == year).astype(int)

    print(f"분석 준비 완료: {df.shape}")
    print(f"사용 가능한 표준화 변수: {[var for var in df.columns if var.endswith('_scaled')]}")

    # 모델 1: 기본 모델
    model1 = smf.ols('MBV ~ sentiment_mean_scaled', data=df).fit()

    # 모델 2: 기업 특성 통제
    control_vars = ['log_assets_scaled']
    if 'debt_to_equity_scaled' in df.columns:
        control_vars.append('debt_to_equity_scaled')
    if 'roa_scaled' in df.columns:
        control_vars.append('roa_scaled')

    formula2 = 'MBV ~ sentiment_mean_scaled + ' + ' + '.join(control_vars)
    model2 = smf.ols(formula2, data=df).fit()

    # 모델 3: 시장 변수 추가
    if 'market_liquidity_scaled' in df.columns:
        formula3 = formula2 + ' + market_liquidity_scaled'
        model3 = smf.ols(formula3, data=df).fit()
    else:
        model3 = model2

    # 모델 4: 시간 고정효과
    year_dummies = [col for col in df.columns if col.startswith('year_')]
    if year_dummies:
        formula4 = formula3.replace('MBV ~', 'MBV ~') + ' + ' + ' + '.join(year_dummies)
        model4 = smf.ols(formula4, data=df).fit()
    else:
        model4 = model3

    models = [model1, model2, model3, model4]
    model_names = ['기본', '기업통제', '시장통제', '시간통제']

    # 결과 비교
    print("\n모델 비교:")
    print("="*80)

    results = []
    for i, (model, name) in enumerate(zip(models, model_names)):
        sentiment_coef = model.params.get('sentiment_mean_scaled', np.nan)
        sentiment_pval = model.pvalues.get('sentiment_mean_scaled', np.nan)

        result = {
            'Model': f'{i+1}. {name}',
            'N': int(model.nobs),
            'R²': model.rsquared,
            'Adj_R²': model.rsquared_adj,
            'Sentiment_β': sentiment_coef,
            'Sentiment_p': sentiment_pval,
            'AIC': model.aic
        }
        results.append(result)

    results_df = pd.DataFrame(results)
    print(results_df.round(6))

    # 최적 모델 선택
    best_model = min(models, key=lambda x: x.aic)
    best_idx = models.index(best_model)

    print(f"\n최적 모델: {model_names[best_idx]} (AIC 기준)")
    print("="*60)
    print(best_model.summary())

    return best_model, df

def perform_robustness_checks(model, df):
    """강건성 검정"""

    print("\n8. 강건성 검정")
    print("-" * 80)

    # 핵심 결과 추출
    if 'sentiment_mean_scaled' not in model.params.index:
        print("감성 변수가 모델에 없습니다.")
        return False

    main_coef = model.params['sentiment_mean_scaled']
    main_pval = model.pvalues['sentiment_mean_scaled']
    main_tstat = model.tvalues['sentiment_mean_scaled']

    print(f"주요 결과:")
    print(f"   감성 계수: {main_coef:.6f}")
    print(f"   t-통계량: {main_tstat:.4f}")
    print(f"   p-value: {main_pval:.6f}")

    # 1. 이상치 제거 후 재분석
    print(f"\n1) 이상치 제거 후 재분석:")

    residuals = model.resid
    outliers = np.abs(zscore(residuals)) > 2.5

    if outliers.sum() > 0:
        robust_data = df[~outliers].copy()
        print(f"   이상치 제거: {outliers.sum()}개 ({outliers.sum()/len(df)*100:.1f}%)")

        # 동일 모델로 재추정
        robust_formula = 'MBV ~ sentiment_mean_scaled + log_assets_scaled'
        if 'roa_scaled' in robust_data.columns:
            robust_formula += ' + roa_scaled'

        robust_model = smf.ols(robust_formula, data=robust_data).fit()
        robust_coef = robust_model.params.get('sentiment_mean_scaled', np.nan)
        robust_pval = robust_model.pvalues.get('sentiment_mean_scaled', np.nan)

        print(f"   강건성 결과: β={robust_coef:.6f}, p={robust_pval:.6f}")

        # 계수 안정성 확인
        coef_change = abs(robust_coef - main_coef) / abs(main_coef) if main_coef != 0 else 0
        print(f"   계수 변화율: {coef_change*100:.1f}%")

        if coef_change < 0.3:
            print("   → 결과 안정적")
        else:
            print("   → 이상치 영향 존재")

    # 2. HAC 표준오차
    print(f"\n2) HAC 강건 표준오차:")
    try:
        hac_results = model.get_robustcov_results('HAC')
        hac_pval = hac_results.pvalues.get('sentiment_mean_scaled', np.nan)
        print(f"   HAC p-value: {hac_pval:.6f}")
    except:
        print("   HAC 계산 실패")

    # 3. 서브샘플 분석
    print(f"\n3) 서브샘플 분석:")

    # 대기업만 분석 (상위 50%)
    median_assets = df['log_assets'].median()
    large_firms = df[df['log_assets'] >= median_assets].copy()

    if len(large_firms) >= 30:
        sub_formula = 'MBV ~ sentiment_mean_scaled + log_assets_scaled'
        sub_model = smf.ols(sub_formula, data=large_firms).fit()
        sub_coef = sub_model.params.get('sentiment_mean_scaled', np.nan)
        sub_pval = sub_model.pvalues.get('sentiment_mean_scaled', np.nan)

        print(f"   대기업 서브샘플 (N={len(large_firms)}): β={sub_coef:.6f}, p={sub_pval:.6f}")

    return True

def final_conclusion(model, df):
    """최종 결론 및 해석"""

    print("\n9. 최종 결론")
    print("="*80)

    if 'sentiment_mean_scaled' not in model.params.index:
        print("분석 실패: 감성 변수가 최종 모델에 포함되지 않음")
        return False

    coef = model.params['sentiment_mean_scaled']
    pval = model.pvalues['sentiment_mean_scaled']
    tstat = model.tvalues['sentiment_mean_scaled']
    conf_int = model.conf_int().loc['sentiment_mean_scaled']

    print(f"핵심 가설검정 결과:")
    print(f"H1: 뉴스 감성점수가 높을수록 기업가치(MBV)가 높다")
    print(f"")
    print(f"통계적 결과:")
    print(f"   - 회귀계수(β): {coef:.6f}")
    print(f"   - t-통계량: {tstat:.4f}")
    print(f"   - p-value: {pval:.6f}")
    print(f"   - 95% 신뢰구간: [{conf_int[0]:.6f}, {conf_int[1]:.6f}]")
    print(f"   - 표본 크기: {int(model.nobs)}")
    print(f"   - R-squared: {model.rsquared:.4f}")

    # 통계적 유의성 판정
    if pval < 0.01:
        significance = "1% 수준에서 통계적으로 유의"
        conclusion = "H1 강력 지지"
        success = True
    elif pval < 0.05:
        significance = "5% 수준에서 통계적으로 유의"
        conclusion = "H1 지지"
        success = True
    elif pval < 0.10:
        significance = "10% 수준에서 통계적으로 유의"
        conclusion = "H1 약한 지지"
        success = True
    else:
        significance = "통계적으로 유의하지 않음"
        conclusion = "H1 기각"
        success = False

    print(f"")
    print(f"가설검정 결론:")
    print(f"   - 통계적 유의성: {significance}")
    print(f"   - 연구 결론: {conclusion}")

    # 경제적 해석 (유의한 경우만)
    if success:
        print(f"")
        print(f"경제적 해석:")

        # 표준화 계수 해석
        sentiment_std = df['sentiment_mean'].std()
        mbv_std = df['MBV'].std()

        print(f"   - 감성점수 1 표준편차({sentiment_std:.3f}) 증가 시")
        print(f"   - MBV가 {coef:.4f} 증가 (표준화 기준)")

        # 실제 단위 해석
        actual_effect = coef * mbv_std / sentiment_std
        print(f"   - 실제 단위: 감성점수 0.1 증가 → MBV {actual_effect*0.1:.4f} 증가")

        if coef > 0:
            print(f"   - 해석: 긍정적 뉴스 감성이 기업가치를 향상시킴")
        else:
            print(f"   - 해석: 부정적 결과 (이론과 불일치)")

    print(f"")
    print(f"연구의 한계:")
    print(f"   - 표본 크기: {int(model.nobs)}개 (추가 확대 권장)")
    print(f"   - 내생성: 역인과관계 가능성")
    print(f"   - 생략변수: 관찰되지 않은 기업 특성")

    return success

def main_analysis():
    """메인 분석 실행"""

    print("정교한 통계 분석 시작")
    print("="*100)

    try:
        # 1. 데이터 로딩 및 탐색
        news_df, financial_df, market_df = load_and_explore_data()

        # 2. 데이터 정제
        news_clean, financial_clean, market_clean = clean_and_standardize_data(
            news_df, financial_df, market_df
        )

        # 3. 변수 생성
        sentiment_features = create_sentiment_aggregates(news_clean)
        financial_features = create_financial_ratios(financial_clean)

        # 4. 데이터 병합
        final_dataset = merge_all_data(sentiment_features, financial_features, market_clean)

        # 표본 크기 확인
        if len(final_dataset) < 50:
            print(f"경고: 표본 크기가 부족합니다 (N={len(final_dataset)})")
            print("최소 50개 이상의 관측치가 권장됩니다.")
            return False

        # 5. 데이터 품질 검증
        validate_data_quality(final_dataset)

        # 6. 회귀분석 실행
        best_model, analysis_data = run_hierarchical_regression(final_dataset)

        # 7. 강건성 검정
        perform_robustness_checks(best_model, analysis_data)

        # 8. 최종 결론
        success = final_conclusion(best_model, analysis_data)

        print(f"\n" + "="*100)
        if success:
            print("분석 성공: 통계적으로 유의한 결과 도출")
            print("뉴스 감성이 기업가치에 미치는 영향 실증적으로 확인")
        else:
            print("분석 완료: 통계적 유의성 미달")
            print("가설 지지 증거 부족 - 추가 연구 필요")
        print("="*100)

        return success

    except Exception as e:
        print(f"분석 중 오류 발생: {str(e)}")
        import traceback
        traceback.print_exc()
        return False

# 분석 실행
if __name__ == "__main__":
    success = main_analysis()

    if success:
        print("\n연구 성과: 학술 논문 작성 및 정책 제언 가능")
        print("신뢰할 수 있는 실증 결과 확보")
    else:
        print("\n연구 개선 방향:")
        print("1. 표본 크기 확대")
        print("2. 추가 통제변수 도입")
        print("3. 내생성 문제 해결 (도구변수 등)")
        print("4. 산업별 세분화 분석")
        print("5. 시간 지연 효과 검토")

정교한 뉴스 감성-기업가치 분석 (엄밀한 통계적 접근)
목표: 신뢰할 수 있는 p-value < 0.1 결과 도출
정교한 통계 분석 시작

1. 데이터 로딩 및 기초 탐색
--------------------------------------------------------------------------------
뉴스 데이터: (12789, 18)
재무 데이터: (18701, 11)
시가총액 데이터: (21892, 9)

감성점수 분포:
평균: 0.2927, 표준편차: 0.3192
범위: [-0.771, 0.755]

2. 데이터 정제 및 표준화
--------------------------------------------------------------------------------
2-1. 뉴스 데이터 정제
   정제 후: (12745, 20)
2-2. 재무 데이터 정제
   정제 후: (16793, 12)
2-3. 시가총액 데이터 정제
   정제 후: (5355, 6)

3. 뉴스 감성 집계 (기업-연도별)
--------------------------------------------------------------------------------
   집계 결과: 1947 → 581 (뉴스 5개 이상)
   평균 뉴스 개수: 18.0

4. 재무 비율 및 지표 생성
--------------------------------------------------------------------------------
   재무 지표 생성 완료: (16793, 23)

5. 데이터 병합 및 MBV 계산
--------------------------------------------------------------------------------
   재무-시장 병합: (15046, 27)
   최종 병합: (406, 40)
   MBV 계산 완료: (396, 43)
   MBV 분포: 평균=2.215, 중위값=1.766
   MBV 범위: [0.271, 1

### 가설용 테이블 작성

#### 실제 HTML과 그래프로 표 생

In [24]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler, RobustScaler
from scipy.stats import pearsonr, spearmanr, zscore, jarque_bera
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan, het_white
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.sandwich_covariance import cov_hac
import warnings
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

# 새로 추가된 라이브러리들 (논문용 표 시각화)
import matplotlib.patches as patches
from matplotlib.table import Table
import matplotlib.font_manager as fm
from io import BytesIO
import base64

warnings.filterwarnings('ignore')

# ===============================
# 학술표준 논문용 표 생성 함수들
# ===============================

def stars(p):
    """유의성 표시"""
    return '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ('†' if p < 0.10 else '')))

def format_coef_with_pvalue(coef, se, pval):
    """계수와 p값을 학술표준 형식으로 포매팅"""
    star = stars(pval)
    return f"{coef:.3f}{star}\n({se:.3f})\n[{pval:.3f}]"

def extract_table_block_enhanced(res, var_order=None, var_rename=None):
    """회귀분석 결과에서 계수/표준오차/p값 추출 (향상된 버전)"""
    params = res.params.copy()
    bse = res.bse.copy()
    pvals = res.pvalues.copy()

    if var_order is None:
        var_order = [v for v in params.index if v.lower() not in ('const', '_cons', 'intercept')]

    def pretty_name(name):
        if var_rename and name in var_rename:
            return var_rename[name]
        return name

    rows = {}
    for v in var_order:
        if v in params.index:
            coef = params[v]
            se = bse[v]
            pv = pvals[v]
            rows[pretty_name(v)] = format_coef_with_pvalue(coef, se, pv)
        else:
            rows[pretty_name(v)] = ""

    return pd.Series(rows, name="")

def make_regression_table_enhanced(results_list, model_names=None, var_order=None, var_rename=None, add_rows=None, show_adj_r2=True):
    """학술표준 논문용 회귀분석 표 생성"""

    if model_names is None:
        model_names = [f"({i+1})" for i in range(len(results_list))]

    # 계수 블록 생성
    cols = []
    for res in results_list:
        col = extract_table_block_enhanced(res, var_order=var_order, var_rename=var_rename)
        cols.append(col)

    body = pd.concat(cols, axis=1)
    body.columns = model_names

    # 하단 통계 정보
    foot_rows = []

    # 관측치 수
    n_list = [int(res.nobs) for res in results_list]
    foot_rows.append(pd.Series({m: f"{n:,d}" for m, n in zip(model_names, n_list)}, name="Observations"))

    # R-squared
    if show_adj_r2:
        adj_r2_list = [getattr(res, "rsquared_adj", np.nan) for res in results_list]
        foot_rows.append(pd.Series({m: f"{r:.3f}" for m, r in zip(model_names, adj_r2_list)}, name="Adj. R-squared"))
    else:
        r2_list = [res.rsquared for res in results_list]
        foot_rows.append(pd.Series({m: f"{r:.3f}" for m, r in zip(model_names, r2_list)}, name="R-squared"))

    # F-통계량
    f_list = [getattr(res, "fvalue", np.nan) for res in results_list]
    foot_rows.append(pd.Series({m: f"{f:.2f}" for m, f in zip(model_names, f_list)}, name="F-statistic"))

    # 사용자 정의 메타정보
    if add_rows:
        for k, v in add_rows.items():
            if isinstance(v, list):
                ser = pd.Series({m: v[i] for i, m in enumerate(model_names)}, name=k)
            else:
                ser = pd.Series({m: v for m in model_names}, name=k)
            foot_rows.append(ser)

    # 전체 표 결합
    foot_df = pd.DataFrame(foot_rows)
    table = pd.concat([body, foot_df], axis=0)

    return table

def create_academic_standard_html(table, title="Regression Results"):
    """APA/AER 표준을 따르는 학술논문용 HTML 표 생성"""

    html_templates = []
    font_sizes = ['11px', '12px', '13px']
    size_labels = ['Small', 'Medium', 'Large']

    for i, (font_size, size_label) in enumerate(zip(font_sizes, size_labels)):

        html = f"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{title} - {size_label}</title>
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Times+New+Roman:wght@400;700&display=swap');

        body {{
            font-family: "Times New Roman", Times, serif;
            margin: 30px;
            background-color: white;
            line-height: 1.2;
        }}

        .table-container {{
            max-width: 900px;
            margin: 0 auto;
            background-color: white;
            padding: 20px;
        }}

        .table-title {{
            text-align: center;
            font-size: {str(int(font_size.replace('px', '')) + 3)}px;
            font-weight: bold;
            margin-bottom: 20px;
            color: #000;
            font-family: "Times New Roman", Times, serif;
        }}

        .regression-table {{
            width: 100%;
            border-collapse: collapse;
            margin: 0 auto;
            font-size: {font_size};
            background-color: white;
            font-family: "Times New Roman", Times, serif;
        }}

        .regression-table th {{
            background-color: white;
            border-top: 2px solid #000;
            border-bottom: 1px solid #000;
            border-left: none;
            border-right: none;
            padding: 8px 6px;
            text-align: center;
            font-weight: bold;
            color: #000;
            height: 40px;
            vertical-align: middle;
        }}

        .regression-table th:first-child {{
            text-align: left;
            padding-left: 0px;
        }}

        .regression-table td {{
            border: none;
            padding: 6px 6px;
            text-align: center;
            background-color: white;
            color: #000;
            height: 60px;
            vertical-align: middle;
            line-height: 1.1;
        }}

        .regression-table .var-name {{
            text-align: left;
            font-weight: normal;
            background-color: white;
            padding-left: 0px;
        }}

        .regression-table .stats-section {{
            border-top: 1px solid #000;
        }}

        .regression-table .stats-row td {{
            height: 30px;
            padding: 4px 6px;
        }}

        .regression-table .final-row {{
            border-bottom: 2px solid #000;
        }}

        .footnote {{
            font-size: {str(int(font_size.replace('px', '')) - 1)}px;
            margin-top: 15px;
            color: #000;
            line-height: 1.4;
            font-family: "Times New Roman", Times, serif;
        }}

        .coefficient {{
            font-weight: normal;
        }}

        .std-error {{
            font-style: italic;
            color: #333;
        }}

        .p-value {{
            font-size: {str(int(font_size.replace('px', '')) - 1)}px;
            color: #666;
        }}

        @media print {{
            body {{
                margin: 0;
            }}
            .table-container {{
                max-width: none;
            }}
        }}

        .interpretation-section {{
            margin-top: 30px;
            padding: 20px;
            background-color: #f9f9f9;
            border-left: 4px solid #333;
        }}

        .interpretation-title {{
            font-size: {str(int(font_size.replace('px', '')) + 1)}px;
            font-weight: bold;
            margin-bottom: 15px;
            color: #000;
        }}

        .interpretation-content {{
            font-size: {font_size};
            line-height: 1.6;
            color: #333;
        }}

        .indicator-explanation {{
            margin: 10px 0;
            padding: 8px;
            background-color: white;
            border-left: 2px solid #666;
        }}

        .indicator-name {{
            font-weight: bold;
            color: #000;
        }}
    </style>
</head>
<body>
    <div class="table-container">
        <div class="table-title">{title}</div>
        <table class="regression-table">
            <thead>
                <tr>
                    <th style="text-align: left;">Variable</th>
"""

        # 헤더 생성
        for col in table.columns:
            html += f'                    <th>{col.replace("\\n", "<br>")}</th>\n'

        html += """                </tr>
            </thead>
            <tbody>
"""

        # 데이터 행 생성
        stats_start_idx = None
        rows_list = list(table.iterrows())

        for idx, (row_name, row_data) in enumerate(rows_list):
            # 통계 섹션 시작점 감지
            if row_name in ['Observations', 'R-squared', 'Adj. R-squared', 'F-statistic'] and stats_start_idx is None:
                stats_start_idx = idx

            # 클래스 결정
            row_class = ''
            if stats_start_idx is not None and idx == stats_start_idx:
                row_class = 'stats-section'
            if stats_start_idx is not None and idx >= stats_start_idx:
                row_class += ' stats-row'
            if idx == len(rows_list) - 1:  # 마지막 행
                row_class += ' final-row'

            html += f'                <tr class="{row_class.strip()}">\n'
            html += f'                    <td class="var-name">{row_name}</td>\n'

            for col_name, cell_value in row_data.items():
                # 줄바꿈 처리
                formatted_value = str(cell_value)

                # 계수/표준오차/p값 형식 처리
                if '\n' in formatted_value and '[' in formatted_value:
                    parts = formatted_value.split('\n')
                    if len(parts) >= 3:
                        coef_part = f'<span class="coefficient">{parts[0]}</span>'
                        se_part = f'<span class="std-error">{parts[1]}</span>'
                        p_part = f'<span class="p-value">{parts[2]}</span>'
                        formatted_value = f"{coef_part}<br>{se_part}<br>{p_part}"
                    else:
                        formatted_value = formatted_value.replace('\n', '<br>')
                else:
                    formatted_value = formatted_value.replace('\n', '<br>')

                html += f'                    <td>{formatted_value}</td>\n'

            html += '                </tr>\n'

        html += """            </tbody>
        </table>

        <div class="footnote">
            <p><strong>Notes:</strong> Standard errors are reported in parentheses, p-values in brackets.
            *** p&lt;0.001, ** p&lt;0.01, * p&lt;0.05, † p&lt;0.10.
            Robust standard errors are used for all specifications.</p>
        </div>

        <!-- 해석 가이드 섹션 -->
        <div class="interpretation-section">
            <div class="interpretation-title">Statistical Indicators Interpretation Guide</div>
            <div class="interpretation-content">

                <div class="indicator-explanation">
                    <div class="indicator-name">News Sentiment:</div>
                    Measures the impact of news sentiment on firm valuation (Market-to-Book ratio).
                    Positive coefficient indicates that positive news sentiment increases firm value.
                </div>

                <div class="indicator-explanation">
                    <div class="indicator-name">Firm Size (log):</div>
                    Natural logarithm of total assets. Controls for firm size effects.
                    Larger firms may have different valuation patterns.
                </div>

                <div class="indicator-explanation">
                    <div class="indicator-name">Leverage:</div>
                    Debt-to-equity ratio. Higher leverage may indicate higher financial risk,
                    potentially affecting firm valuation.
                </div>

                <div class="indicator-explanation">
                    <div class="indicator-name">ROA (Return on Assets):</div>
                    Measures operational efficiency. Higher ROA typically indicates
                    better management performance and should positively affect valuation.
                </div>

                <div class="indicator-explanation">
                    <div class="indicator-name">Market Liquidity:</div>
                    Trading volume measure. Higher liquidity may reduce information asymmetry
                    and affect the relationship between news sentiment and valuation.
                </div>

                <div class="indicator-explanation">
                    <div class="indicator-name">Year Fixed Effects:</div>
                    Controls for time-specific factors affecting all firms in a given year
                    (e.g., macroeconomic conditions, regulatory changes).
                </div>

                <div class="indicator-explanation">
                    <div class="indicator-name">Adj. R-squared:</div>
                    Adjusted coefficient of determination. Indicates the proportion of variance
                    in the dependent variable explained by the model, adjusted for degrees of freedom.
                </div>

                <div class="indicator-explanation">
                    <div class="indicator-name">F-statistic:</div>
                    Tests the overall significance of the regression model.
                    Higher values indicate better model fit.
                </div>

                <div class="indicator-explanation">
                    <div class="indicator-name">Statistical Significance:</div>
                    *** p&lt;0.001 (highly significant), ** p&lt;0.01 (significant),
                    * p&lt;0.05 (significant), † p&lt;0.10 (marginal significance)
                </div>

            </div>
        </div>

    </div>
</body>
</html>"""

        html_templates.append((size_label, html))

    return html_templates

def create_academic_standard_image(table, title="Regression Results"):
    """학술표준을 따르는 논문용 이미지 표 생성"""

    # 폰트 설정 - 시스템에서 Times New Roman 계열 찾기
    font_name = 'serif'  # 기본값
    try:
        available_fonts = [f.name for f in fm.fontManager.ttflist]

        # 우선순위: Times New Roman 계열
        font_candidates = [
            'Times New Roman',
            'Times',
            'Liberation Serif',
            'DejaVu Serif',
            'serif'
        ]

        for font in font_candidates:
            if font in available_fonts:
                font_name = font
                break

        plt.rcParams['font.family'] = font_name
        plt.rcParams['font.serif'] = [font_name]

    except Exception as e:
        print(f"폰트 설정 경고: {e}")
        plt.rcParams['font.family'] = 'serif'

    image_files = []
    font_sizes = [9, 11, 13]
    figsize_list = [(12, 10), (14, 12), (16, 14)]
    size_labels = ['Small', 'Medium', 'Large']

    for i, (font_size, figsize, size_label) in enumerate(zip(font_sizes, figsize_list, size_labels)):

        fig, ax = plt.subplots(figsize=figsize, facecolor='white', dpi=300)
        ax.axis('off')

        # 제목
        fig.suptitle(title, fontsize=font_size+3, fontweight='bold', y=0.95,
                    fontfamily='serif')

        # 표 데이터 준비
        table_data = []
        headers = ['Variable'] + list(table.columns)

        # 데이터 행들
        for row_name, row_data in table.iterrows():
            row = [row_name]
            for col_value in row_data:
                # p값 포함 형식 처리
                formatted_value = str(col_value)
                if '\n' in formatted_value and '[' in formatted_value:
                    # 3줄 형식: 계수\n(표준오차)\n[p값]
                    parts = formatted_value.split('\n')
                    if len(parts) >= 3:
                        # 별표 분리
                        coef_line = parts[0]
                        se_line = parts[1]
                        p_line = parts[2]
                        formatted_value = f"{coef_line}\n{se_line}\n{p_line}"
                row.append(formatted_value)
            table_data.append(row)

        # 표 그리기
        n_rows = len(table_data) + 1  # +1 for header
        n_cols = len(headers)

        # 표 위치 및 크기 설정
        table_ax = fig.add_subplot(111)
        table_ax.axis('off')

        # 균등한 셀 크기
        table_width = 0.85
        table_height = 0.75
        cell_width = table_width / n_cols
        cell_height = table_height / n_rows

        start_x = 0.075
        start_y = 0.8

        # 헤더 그리기
        for j, header in enumerate(headers):
            x_pos = start_x + j * cell_width
            y_pos = start_y

            # 헤더 상단 굵은 선
            if j == 0:
                table_ax.plot([start_x, start_x + table_width], [y_pos + cell_height*0.1, y_pos + cell_height*0.1],
                             'k-', linewidth=2)

            # 헤더 텍스트
            ha = 'left' if j == 0 else 'center'
            table_ax.text(x_pos + (0.01 if j == 0 else cell_width/2), y_pos - cell_height/2,
                         header.replace('\\n', '\n'),
                         ha=ha, va='center', fontsize=font_size+1, fontweight='bold',
                         fontfamily='serif')

            # 헤더 하단 선
            if j == 0:
                table_ax.plot([start_x, start_x + table_width], [y_pos - cell_height*0.9, y_pos - cell_height*0.9],
                             'k-', linewidth=1)

        # 데이터 행 그리기
        stats_start_idx = None
        for i, row in enumerate(table_data):
            # 통계 섹션 감지
            if row[0] in ['Observations', 'R-squared', 'Adj. R-squared', 'F-statistic'] and stats_start_idx is None:
                stats_start_idx = i
                # 통계 섹션 시작 선
                y_line = start_y - (i + 1) * cell_height + cell_height*0.1
                table_ax.plot([start_x, start_x + table_width], [y_line, y_line],
                             'k-', linewidth=1)

            for j, cell_value in enumerate(row):
                x_pos = start_x + j * cell_width
                y_pos = start_y - (i + 1) * cell_height

                # 텍스트 정렬
                ha = 'left' if j == 0 else 'center'
                x_text = x_pos + (0.01 if j == 0 else cell_width/2)

                # 텍스트 크기 조정
                text_size = font_size if stats_start_idx is None or i < stats_start_idx else font_size - 1

                # 다줄 텍스트 처리
                if '\n' in str(cell_value) and j > 0:
                    lines = str(cell_value).split('\n')
                    line_spacing = cell_height / (len(lines) + 1)
                    for k, line in enumerate(lines):
                        line_y = y_pos - cell_height/2 + (len(lines)/2 - k) * line_spacing * 0.7

                        # 스타일 적용
                        weight = 'normal'
                        color = 'black'
                        if k == 1:  # 표준오차
                            color = '#444'
                        elif k == 2:  # p값
                            color = '#666'
                            text_size = font_size - 1

                        table_ax.text(x_text, line_y, line,
                                     ha=ha, va='center', fontsize=text_size,
                                     fontweight=weight, color=color, fontfamily='serif')
                else:
                    table_ax.text(x_text, y_pos - cell_height/2, str(cell_value),
                                 ha=ha, va='center', fontsize=text_size,
                                 fontweight='normal', fontfamily='serif')

        # 마지막 하단 굵은 선
        final_y = start_y - n_rows * cell_height + cell_height*0.1
        table_ax.plot([start_x, start_x + table_width], [final_y, final_y],
                     'k-', linewidth=2)

        # 범례
        footnote_y = final_y - 0.05
        footnote_text = ('Notes: Standard errors in parentheses, p-values in brackets. '
                        '*** p<0.001, ** p<0.01, * p<0.05, † p<0.10. '
                        'Robust standard errors used.')

        table_ax.text(start_x, footnote_y, footnote_text,
                     fontsize=font_size-1, ha='left', va='top',
                     fontfamily='serif', wrap=True)

        # 레이아웃 조정
        plt.tight_layout()
        plt.subplots_adjust(top=0.92, bottom=0.08)

        # 파일 저장
        filename = f'academic_table_{size_label.lower()}.png'
        plt.savefig(filename, dpi=300, bbox_inches='tight',
                   facecolor='white', edgecolor='none', format='png',
                   pad_inches=0.2)

        image_files.append((size_label, filename))
        plt.close()

    return image_files

def save_academic_standard_formats(table, filename_base="academic_regression", title="Effect of News Sentiment on Market-to-Book Value"):
    """학술표준 형식으로 표 저장"""

    print(f"\n📊 학술표준 회귀분석 표:")
    print("="*100)
    print(table)
    print("="*100)

    # 1. 기존 형식들 저장
    try:
        excel_file = f"{filename_base}.xlsx"
        table.to_excel(excel_file, index=True)
        print(f"✓ Excel 저장: {excel_file}")
    except Exception as e:
        print(f"Excel 저장 실패: {e}")

    try:
        csv_file = f"{filename_base}.csv"
        table.to_csv(csv_file, index=True, encoding='utf-8-sig')
        print(f"✓ CSV 저장: {csv_file}")
    except Exception as e:
        print(f"CSV 저장 실패: {e}")

    # 2. 학술표준 HTML 저장
    print(f"\n🎓 학술표준 시각적 표 생성 중...")
    try:
        html_versions = create_academic_standard_html(table, title)

        for size_label, html_content in html_versions:
            html_file = f"{filename_base}_{size_label.lower()}.html"
            with open(html_file, 'w', encoding='utf-8') as f:
                f.write(html_content)
            print(f"✓ 학술표준 HTML ({size_label}): {html_file}")

    except Exception as e:
        print(f"HTML 생성 실패: {e}")

    # 3. 학술표준 이미지 저장
    try:
        image_versions = create_academic_standard_image(table, title)

        for size_label, image_file in image_versions:
            print(f"✓ 학술표준 이미지 ({size_label}): {image_file}")

    except Exception as e:
        print(f"이미지 생성 실패: {e}")

    print(f"\n🏆 학술표준 논문 표 완성!")
    print(f"   📋 특징:")
    print(f"      • APA/AER 저널 스타일 준수")
    print(f"      • 계수 + 표준오차 + p값 모두 표시")
    print(f"      • 균등한 셀 높이")
    print(f"      • Times New Roman 폰트")
    print(f"      • 학술논문 표준 격자선")
    print(f"      • 통계 지표 해석 가이드 포함")
    print(f"   📐 크기: Small, Medium, Large")
    print(f"   📄 형식: HTML (스크린샷용) + PNG (직접삽입용)")

# ===============================
# 테스트용 Mock 데이터 생성
# ===============================

def create_mock_regression_results():
    """Mock 회귀분석 결과 생성"""

    # Mock 데이터 생성
    np.random.seed(42)
    n = 396

    data = pd.DataFrame({
        'MBV': np.random.normal(2.2, 0.8, n),
        'sentiment_mean_scaled': np.random.normal(0, 1, n),
        'log_assets_scaled': np.random.normal(0, 1, n),
        'debt_to_equity_scaled': np.random.normal(0, 1, n),
        'roa_scaled': np.random.normal(0, 1, n),
        'market_liquidity_scaled': np.random.normal(0, 1, n),
        'year_2021': np.random.binomial(1, 0.2, n),
        'year_2022': np.random.binomial(1, 0.2, n),
        'year_2023': np.random.binomial(1, 0.2, n)
    })

    # 회귀모델들 실행
    models = []
    model_names = []

    # 모델 1: 기본
    model1 = smf.ols('MBV ~ sentiment_mean_scaled', data=data).fit(cov_type='HC1')
    models.append(model1)
    model_names.append('Basic')

    # 모델 2: 기업통제
    model2 = smf.ols('MBV ~ sentiment_mean_scaled + log_assets_scaled + debt_to_equity_scaled + roa_scaled', data=data).fit(cov_type='HC1')
    models.append(model2)
    model_names.append('Firm Controls')

    # 모델 3: 시장통제
    model3 = smf.ols('MBV ~ sentiment_mean_scaled + log_assets_scaled + debt_to_equity_scaled + roa_scaled + market_liquidity_scaled', data=data).fit(cov_type='HC1')
    models.append(model3)
    model_names.append('Market Controls')

    # 모델 4: 시간통제
    model4 = smf.ols('MBV ~ sentiment_mean_scaled + log_assets_scaled + debt_to_equity_scaled + roa_scaled + market_liquidity_scaled + year_2021 + year_2022 + year_2023', data=data).fit(cov_type='HC1')
    models.append(model4)
    model_names.append('Year FE')

    return models, model_names

def test_academic_standard_table():
    """학술표준 표 생성 테스트"""

    print("🎓 학술표준 논문 표 생성 테스트")
    print("=" * 60)

    # Mock 데이터 생성
    models, model_names = create_mock_regression_results()

    # 변수 설정
    var_order = [
        'sentiment_mean_scaled',
        'log_assets_scaled',
        'debt_to_equity_scaled',
        'roa_scaled',
        'market_liquidity_scaled',
        'year_2021',
        'year_2022',
        'year_2023'
    ]

    var_rename = {
        'sentiment_mean_scaled': 'News Sentiment',
        'log_assets_scaled': 'Firm Size (log)',
        'debt_to_equity_scaled': 'Leverage',
        'roa_scaled': 'ROA',
        'market_liquidity_scaled': 'Market Liquidity',
        'year_2021': 'Year 2021',
        'year_2022': 'Year 2022',
        'year_2023': 'Year 2023'
    }

    # 모델명
    paper_model_names = []
    for i, name in enumerate(model_names):
        paper_model_names.append(f"({i+1})")

    # 추가 정보
    n_models = len(models)
    add_rows = {
        "Firm Controls": ["No"] + ["Yes"] * (n_models - 1),
        "Market Controls": ["No"] * 2 + ["Yes"] * (n_models - 2),
        "Year Fixed Effects": ["No"] * 3 + ["Yes"] * (n_models - 3),
        "Standard Errors": ["Robust"] * n_models
    }

    # 학술표준 표 생성
    table = make_regression_table_enhanced(
        results_list=models,
        model_names=paper_model_names,
        var_order=var_order,
        var_rename=var_rename,
        add_rows=add_rows,
        show_adj_r2=True
    )

    print("✅ 학술표준 회귀분석 표 생성 완료")

    # 학술표준 형식으로 저장
    save_academic_standard_formats(table, "academic_regression", "Effect of News Sentiment on Market-to-Book Value")

    print("\n🎯 학술표준 테스트 완료!")
    print("생성된 파일들:")
    print("   📄 HTML: academic_regression_small.html, medium.html, large.html")
    print("   🖼️ PNG: academic_table_small.png, medium.png, large.png")
    print("   📊 DATA: academic_regression.xlsx, .csv")

if __name__ == "__main__":
    test_academic_standard_table()

🎓 학술표준 논문 표 생성 테스트
✅ 학술표준 회귀분석 표 생성 완료

📊 학술표준 회귀분석 표:
                                        (1)                       (2)  \
News Sentiment      0.018\n(0.039)\n[0.649]   0.018\n(0.040)\n[0.646]   
Firm Size (log)                               0.020\n(0.037)\n[0.592]   
Leverage                                     -0.003\n(0.039)\n[0.934]   
ROA                                          -0.008\n(0.042)\n[0.843]   
Market Liquidity                                                        
Year 2021                                                               
Year 2022                                                               
Year 2023                                                               
Observations                            396                       396   
Adj. R-squared                       -0.002                    -0.009   
F-statistic                            0.21                      0.14   
Firm Controls                            No                       Yes

### 가설1 TABLE2 (상관관계행렬) 생성

In [50]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler, RobustScaler
from scipy.stats import pearsonr, spearmanr, zscore, jarque_bera
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan, het_white
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.sandwich_covariance import cov_hac
import warnings
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# ===============================
# 논문용 표 생성 함수들
# ===============================

def stars(p):
    """유의성 표시"""
    return '***' if p < 0.01 else ('**' if p < 0.05 else ('*' if p < 0.10 else ''))

def extract_table_block(res, var_order=None, var_rename=None):
    """회귀분석 결과에서 계수/표준오차/유의확률 추출"""
    params = res.params.copy()
    bse = res.bse.copy()
    pvals = res.pvalues.copy()

    if var_order is None:
        var_order = [v for v in params.index if v.lower() not in ('const', '_cons', 'intercept')]

    def pretty_name(name):
        if var_rename and name in var_rename:
            return var_rename[name]
        return name

    rows = {}
    for v in var_order:
        if v in params.index:
            coef = params[v]
            se = bse[v]
            pv = pvals[v]
            rows[pretty_name(v)] = f"{coef:.3f}{stars(pv)}\n({se:.3f})"
        else:
            rows[pretty_name(v)] = ""

    return pd.Series(rows, name="")

def make_regression_table(results_list, model_names=None, var_order=None, var_rename=None, add_rows=None, show_adj_r2=True):
    """논문용 회귀분석 표 생성"""

    if model_names is None:
        model_names = [f"({i+1})" for i in range(len(results_list))]

    # 계수 블록 생성
    cols = []
    for res in results_list:
        col = extract_table_block(res, var_order=var_order, var_rename=var_rename)
        cols.append(col)

    body = pd.concat(cols, axis=1)
    body.columns = model_names

    # 하단 통계 정보
    foot_rows = []

    # 관측치 수
    n_list = [int(res.nobs) for res in results_list]
    foot_rows.append(pd.Series({m: f"{n:,d}" for m, n in zip(model_names, n_list)}, name="Observations"))

    # R-squared
    if show_adj_r2:
        adj_r2_list = [getattr(res, "rsquared_adj", np.nan) for res in results_list]
        foot_rows.append(pd.Series({m: f"{r:.3f}" for m, r in zip(model_names, adj_r2_list)}, name="Adj. R-squared"))
    else:
        r2_list = [res.rsquared for res in results_list]
        foot_rows.append(pd.Series({m: f"{r:.3f}" for m, r in zip(model_names, r2_list)}, name="R-squared"))

    # F-통계량
    f_list = [getattr(res, "fvalue", np.nan) for res in results_list]
    foot_rows.append(pd.Series({m: f"{f:.2f}" for m, f in zip(model_names, f_list)}, name="F-statistic"))

    # 사용자 정의 메타정보
    if add_rows:
        for k, v in add_rows.items():
            if isinstance(v, list):
                ser = pd.Series({m: v[i] for i, m in enumerate(model_names)}, name=k)
            else:
                ser = pd.Series({m: v for m in model_names}, name=k)
            foot_rows.append(ser)

    # 전체 표 결합
    foot_df = pd.DataFrame(foot_rows)
    table = pd.concat([body, foot_df], axis=0)

    return table

def save_all_formats(table, filename_base="sentiment_mbv_results"):
    """표를 모든 형식으로 저장"""

    print(f"\n논문용 회귀분석 표:")
    print("="*100)
    print(table)
    print("="*100)

    # Excel 저장
    try:
        excel_file = f"{filename_base}.xlsx"
        table.to_excel(excel_file, index=True)
        print(f"✓ Excel 저장: {excel_file}")
    except Exception as e:
        print(f"Excel 저장 실패: {e}")

    # CSV 저장
    try:
        csv_file = f"{filename_base}.csv"
        table.to_csv(csv_file, index=True, encoding='utf-8-sig')
        print(f"✓ CSV 저장: {csv_file}")
    except Exception as e:
        print(f"CSV 저장 실패: {e}")

    # LaTeX 저장
    try:
        latex_code = table.to_latex(
            escape=False,
            column_format="l" + "c" * len(table.columns),
            caption="Effect of News Sentiment on Market-to-Book Value",
            label="tab:sentiment_mbv"
        )

        latex_file = f"{filename_base}.tex"
        with open(latex_file, 'w', encoding='utf-8') as f:
            f.write(latex_code)
        print(f"✓ LaTeX 저장: {latex_file}")

        print(f"\nLaTeX 코드:")
        print("-" * 60)
        print(latex_code)

    except Exception as e:
        print(f"LaTeX 저장 실패: {e}")

# ===============================
# 수정된 분석 함수들 (결과 반환)
# ===============================

def load_and_explore_data():
    """데이터 로딩 및 기초 탐색"""

    print("\n1. 데이터 로딩 및 기초 탐색")
    print("-" * 80)

    # 1-1. 뉴스 감성 데이터
    news_df = pd.read_csv('/content/뉴스감성분석(개선판_키워드3개기사만)_20250922_021928.csv', encoding='utf-8-sig')
    print(f"뉴스 데이터: {news_df.shape}")

    # 1-2. 재무 데이터
    financial_df = pd.read_excel('/content/dart_financial_merged(약700개_10년치)재무재표정보).xlsx')
    print(f"재무 데이터: {financial_df.shape}")

    # 1-3. 시가총액 데이터
    market_df = pd.read_csv('/content/market_table_quarterly(KRX시가총액정보_상장주식수_dart기준기업).csv', encoding='utf-8-sig')
    print(f"시가총액 데이터: {market_df.shape}")

    # 기초 통계 확인
    print(f"\n감성점수 분포:")
    print(f"평균: {news_df['감성점수'].mean():.4f}, 표준편차: {news_df['감성점수'].std():.4f}")
    print(f"범위: [{news_df['감성점수'].min():.3f}, {news_df['감성점수'].max():.3f}]")

    return news_df, financial_df, market_df

def clean_and_standardize_data(news_df, financial_df, market_df):
    """데이터 정제 및 표준화"""

    print("\n2. 데이터 정제 및 표준화")
    print("-" * 80)

    # 2-1. 뉴스 데이터 정제
    print("2-1. 뉴스 데이터 정제")
    news_clean = news_df.copy()

    # 종목코드 표준화 (6자리 숫자로 통일)
    news_clean['종목코드'] = pd.to_numeric(news_clean['종목코드'], errors='coerce')
    news_clean = news_clean.dropna(subset=['종목코드'])
    news_clean['종목코드'] = news_clean['종목코드'].astype(int)

    # 날짜 처리
    news_clean['기사날짜'] = pd.to_datetime(news_clean['기사날짜'], errors='coerce')
    news_clean = news_clean.dropna(subset=['기사날짜'])
    news_clean['year'] = news_clean['기사날짜'].dt.year
    news_clean['quarter'] = news_clean['기사날짜'].dt.quarter

    # 감성 데이터 이상치 제거 (표준 3시그마 규칙)
    for col in ['감성점수', '긍정확률', '부정확률', '신뢰도']:
        mean_val = news_clean[col].mean()
        std_val = news_clean[col].std()
        news_clean = news_clean[
            (news_clean[col] >= mean_val - 3*std_val) &
            (news_clean[col] <= mean_val + 3*std_val)
        ]

    print(f"   정제 후: {news_clean.shape}")

    # 2-2. 재무 데이터 정제
    print("2-2. 재무 데이터 정제")
    financial_clean = financial_df.copy()

    # 종목코드 표준화
    financial_clean['종목코드'] = financial_clean['종목코드'].astype(int)
    financial_clean['year'] = financial_clean['bsns_year']

    # 재무 데이터 필터링 (기본 조건)
    financial_clean = financial_clean[
        (financial_clean['자산총계'] > 0) &
        (financial_clean['자본총계'] > 0) &
        (financial_clean['부채총계'] >= 0) &
        (financial_clean['당기순이익'].notna())
    ]

    # 극단 이상치 제거 (상하위 1%)
    for col in ['자산총계', '자본총계', '부채총계', '매출액']:
        if col in financial_clean.columns:
            q01 = financial_clean[col].quantile(0.01)
            q99 = financial_clean[col].quantile(0.99)
            financial_clean = financial_clean[
                (financial_clean[col] >= q01) & (financial_clean[col] <= q99)
            ]

    print(f"   정제 후: {financial_clean.shape}")

    # 2-3. 시가총액 데이터 정제
    print("2-3. 시가총액 데이터 정제")
    market_clean = market_df.copy()

    # 종목코드 표준화
    market_clean['종목코드'] = market_clean['ticker'].astype(int)

    # 연도 추출
    market_clean['year'] = market_clean['period_yyyyq'].str[:4].astype(int)
    market_clean['quarter'] = market_clean['period_yyyyq'].str[-1:].astype(int)

    # 시가총액 유효성 체크
    market_clean = market_clean[
        (market_clean['market_cap_krw'] > 0) &
        (market_clean['shares_outstanding'] > 0)
    ]

    # 연도별 평균 계산 (분기 데이터 → 연간 데이터)
    market_yearly = market_clean.groupby(['종목코드', 'year']).agg({
        'market_cap_krw': 'mean',
        'shares_outstanding': 'mean',
        'volume': 'mean',
        'value_traded': 'mean'
    }).reset_index()

    print(f"   정제 후: {market_yearly.shape}")

    return news_clean, financial_clean, market_yearly

def create_sentiment_aggregates(news_df):
    """뉴스 감성 데이터 기업-연도별 집계"""

    print("\n3. 뉴스 감성 집계 (기업-연도별)")
    print("-" * 80)

    # 기업-연도별 집계 (최소 5개 기사 이상)
    sentiment_agg = news_df.groupby(['종목코드', 'year']).agg({
        '감성점수': ['mean', 'std', 'count', 'median'],
        '긍정확률': ['mean', 'std'],
        '부정확률': ['mean', 'std'],
        '신뢰도': ['mean', 'std']
    }).round(6)

    # 컬럼명 정리
    sentiment_agg.columns = [
        'sentiment_mean', 'sentiment_std', 'news_count', 'sentiment_median',
        'positive_mean', 'positive_std', 'negative_mean', 'negative_std',
        'confidence_mean', 'confidence_std'
    ]

    sentiment_agg = sentiment_agg.reset_index()

    # 뉴스 개수 필터링 (신뢰도 확보)
    min_news = 5
    sentiment_filtered = sentiment_agg[sentiment_agg['news_count'] >= min_news].copy()

    # 추가 지표 생성
    sentiment_filtered['sentiment_volatility'] = sentiment_filtered['sentiment_std'].fillna(0)
    sentiment_filtered['confidence_weighted_sentiment'] = (
        sentiment_filtered['sentiment_mean'] * sentiment_filtered['confidence_mean']
    )
    sentiment_filtered['news_intensity'] = sentiment_filtered['news_count'] / 365  # 일평균

    print(f"   집계 결과: {len(sentiment_agg)} → {len(sentiment_filtered)} (뉴스 {min_news}개 이상)")
    print(f"   평균 뉴스 개수: {sentiment_filtered['news_count'].mean():.1f}")

    return sentiment_filtered

def create_financial_ratios(financial_df):
    """재무 비율 및 지표 생성"""

    print("\n4. 재무 비율 및 지표 생성")
    print("-" * 80)

    financial_enhanced = financial_df.copy()

    # 기본 재무 비율
    financial_enhanced['debt_to_equity'] = financial_enhanced['부채총계'] / financial_enhanced['자본총계']
    financial_enhanced['debt_to_assets'] = financial_enhanced['부채총계'] / financial_enhanced['자산총계']
    financial_enhanced['roa'] = financial_enhanced['당기순이익'] / financial_enhanced['자산총계']
    financial_enhanced['roe'] = financial_enhanced['당기순이익'] / financial_enhanced['자본총계']

    # 규모 지표
    financial_enhanced['log_assets'] = np.log(financial_enhanced['자산총계'])
    financial_enhanced['log_equity'] = np.log(financial_enhanced['자본총계'])

    # 매출 기반 비율 (매출액이 있는 경우)
    financial_enhanced['has_revenue'] = financial_enhanced['매출액'].notna() & (financial_enhanced['매출액'] > 0)
    financial_enhanced['asset_turnover'] = np.where(
        financial_enhanced['has_revenue'],
        financial_enhanced['매출액'] / financial_enhanced['자산총계'],
        np.nan
    )

    # 성장률 계산 (기업별 연도별)
    growth_data = []
    for ticker in financial_enhanced['종목코드'].unique():
        company_data = financial_enhanced[financial_enhanced['종목코드'] == ticker].sort_values('year')
        if len(company_data) >= 2:
            company_data['asset_growth'] = company_data['자산총계'].pct_change()
            company_data['equity_growth'] = company_data['자본총계'].pct_change()
        growth_data.append(company_data)

    financial_final = pd.concat(growth_data, ignore_index=True)

    # 산업 규모 구분 (자산 기준 4분위)
    financial_final['size_quartile'] = pd.qcut(
        financial_final['자산총계'],
        q=4,
        labels=['Small', 'Medium_Small', 'Medium_Large', 'Large'],
        duplicates='drop'
    )

    print(f"   재무 지표 생성 완료: {financial_final.shape}")

    return financial_final

def merge_all_data(sentiment_df, financial_df, market_df):
    """전체 데이터 병합 및 MBV 계산"""

    print("\n5. 데이터 병합 및 MBV 계산")
    print("-" * 80)

    # 1단계: 재무 + 시장 데이터
    fin_market = pd.merge(
        financial_df,
        market_df,
        on=['종목코드', 'year'],
        how='inner'
    )
    print(f"   재무-시장 병합: {fin_market.shape}")

    # 2단계: 감성 데이터 추가
    final_data = pd.merge(
        fin_market,
        sentiment_df,
        on=['종목코드', 'year'],
        how='inner'
    )
    print(f"   최종 병합: {final_data.shape}")

    if len(final_data) < 100:
        print(f"경고: 표본 크기가 작습니다 (N={len(final_data)}). 최소 100개 권장.")

    # MBV 계산 및 이상치 처리
    final_data['MBV'] = final_data['market_cap_krw'] / final_data['자본총계']

    # MBV 이상치 제거 (1-99% 범위)
    mbv_q01 = final_data['MBV'].quantile(0.01)
    mbv_q99 = final_data['MBV'].quantile(0.99)
    final_data = final_data[
        (final_data['MBV'] >= mbv_q01) & (final_data['MBV'] <= mbv_q99)
    ]

    # 로그 변환 (정규성 개선)
    final_data['log_MBV'] = np.log(final_data['MBV'])

    # 추가 시장 지표
    final_data['market_liquidity'] = np.log(final_data['value_traded'] + 1)

    print(f"   MBV 계산 완료: {final_data.shape}")
    print(f"   MBV 분포: 평균={final_data['MBV'].mean():.3f}, 중위값={final_data['MBV'].median():.3f}")
    print(f"   MBV 범위: [{final_data['MBV'].min():.3f}, {final_data['MBV'].max():.3f}]")

    return final_data

def validate_data_quality(df):
    """데이터 품질 검증"""

    print("\n6. 데이터 품질 검증")
    print("-" * 80)

    # 기본 통계
    print(f"최종 분석 데이터: {df.shape}")
    print(f"기업 수: {df['종목코드'].nunique()}")
    print(f"연도 범위: {df['year'].min()}-{df['year'].max()}")
    print(f"기업당 평균 관측치: {len(df) / df['종목코드'].nunique():.1f}")

    # 핵심 변수 상관관계
    key_vars = ['sentiment_mean', 'MBV', 'log_assets', 'roa', 'debt_to_equity']
    available_vars = [var for var in key_vars if var in df.columns]

    if len(available_vars) >= 2:
        print(f"\n핵심 변수 상관관계:")
        corr_matrix = df[available_vars].corr()
        print(corr_matrix.round(4))

    # 감성-MBV 기초 관계
    if 'sentiment_mean' in df.columns:
        corr_basic, p_basic = pearsonr(df['sentiment_mean'], df['MBV'])
        print(f"\n기초 상관관계 (감성-MBV): {corr_basic:.4f} (p={p_basic:.4f})")

    return True

def run_hierarchical_regression(df):
    """위계적 회귀분석 실행 - 모델들 반환"""

    print("\n7. 위계적 회귀분석")
    print("-" * 80)

    # 변수 표준화
    scaler = StandardScaler()

    continuous_vars = ['sentiment_mean', 'log_assets', 'debt_to_equity', 'roa', 'market_liquidity']
    available_vars = [var for var in continuous_vars if var in df.columns]

    for var in available_vars:
        df[f'{var}_scaled'] = scaler.fit_transform(df[[var]])

    # 연도 더미 (중요한 연도만)
    year_counts = df['year'].value_counts()
    major_years = year_counts[year_counts >= 10].index  # 관측치 10개 이상 연도만

    for year in major_years[1:]:  # 첫 번째 연도는 기준
        df[f'year_{year}'] = (df['year'] == year).astype(int)

    print(f"분석 준비 완료: {df.shape}")
    print(f"사용 가능한 표준화 변수: {[var for var in df.columns if var.endswith('_scaled')]}")

    # 모델들 실행
    models = []
    model_names = []

    # 모델 1: 기본 모델
    try:
        model1 = smf.ols('MBV ~ sentiment_mean_scaled', data=df).fit(cov_type='HC1')
        models.append(model1)
        model_names.append('기본')
    except Exception as e:
        print(f"모델 1 실행 실패: {e}")

    # 모델 2: 기업 특성 통제
    try:
        control_vars = ['log_assets_scaled']
        if 'debt_to_equity_scaled' in df.columns:
            control_vars.append('debt_to_equity_scaled')
        if 'roa_scaled' in df.columns:
            control_vars.append('roa_scaled')

        formula2 = 'MBV ~ sentiment_mean_scaled + ' + ' + '.join(control_vars)
        model2 = smf.ols(formula2, data=df).fit(cov_type='HC1')
        models.append(model2)
        model_names.append('기업통제')
    except Exception as e:
        print(f"모델 2 실행 실패: {e}")

    # 모델 3: 시장 변수 추가
    try:
        if 'market_liquidity_scaled' in df.columns:
            formula3 = formula2 + ' + market_liquidity_scaled'
            model3 = smf.ols(formula3, data=df).fit(cov_type='HC1')
            models.append(model3)
            model_names.append('시장통제')
    except Exception as e:
        print(f"모델 3 실행 실패: {e}")

    # 모델 4: 시간 고정효과
    try:
        year_dummies = [col for col in df.columns if col.startswith('year_')]
        if year_dummies and len(models) > 0:
            # 마지막 성공한 모델의 공식에 연도 더미 추가
            last_formula = models[-1].model.formula if hasattr(models[-1].model, 'formula') else 'MBV ~ sentiment_mean_scaled + log_assets_scaled + debt_to_equity_scaled + roa_scaled + market_liquidity_scaled'
            formula4 = last_formula + ' + ' + ' + '.join(year_dummies)
            model4 = smf.ols(formula4, data=df).fit(cov_type='HC1')
            models.append(model4)
            model_names.append('시간통제')
    except Exception as e:
        print(f"모델 4 실행 실패: {e}")

    if not models:
        print("모든 모델 실행 실패")
        return None, None, None

    # 결과 비교
    print("\n모델 비교:")
    print("="*80)

    results = []
    for i, (model, name) in enumerate(zip(models, model_names)):
        sentiment_coef = model.params.get('sentiment_mean_scaled', np.nan)
        sentiment_pval = model.pvalues.get('sentiment_mean_scaled', np.nan)

        result = {
            'Model': f'{i+1}. {name}',
            'N': int(model.nobs),
            'R²': model.rsquared,
            'Adj_R²': model.rsquared_adj,
            'Sentiment_β': sentiment_coef,
            'Sentiment_p': sentiment_pval,
            'AIC': model.aic
        }
        results.append(result)

    results_df = pd.DataFrame(results)
    print(results_df.round(6))

    # 최적 모델 선택
    best_model = min(models, key=lambda x: x.aic)
    best_idx = models.index(best_model)

    print(f"\n최적 모델: {model_names[best_idx]} (AIC 기준)")
    print("="*60)
    print(best_model.summary())

    return models, model_names, best_model

def create_paper_table(models, model_names):
    """논문용 표 생성 및 저장"""

    print("\n8. 논문용 표 생성")
    print("-" * 80)

    # 변수 순서 및 이름 설정
    var_order = [
        'sentiment_mean_scaled',
        'log_assets_scaled',
        'debt_to_equity_scaled',
        'roa_scaled',
        'market_liquidity_scaled'
    ]

    # 연도 더미 변수 추가 (첫 번째 모델에서 확인)
    year_vars = []
    if models:
        for param in models[-1].params.index:  # 마지막 모델에서 연도 변수 확인
            if param.startswith('year_'):
                year_vars.append(param)
        year_vars.sort()
        var_order.extend(year_vars)

    var_rename = {
        'sentiment_mean_scaled': 'News Sentiment',
        'log_assets_scaled': 'Firm Size (log)',
        'debt_to_equity_scaled': 'Leverage',
        'roa_scaled': 'ROA',
        'market_liquidity_scaled': 'Market Liquidity'
    }

    # 연도 변수명 매핑
    for var in year_vars:
        year_num = var.replace('year_', '')
        var_rename[var] = f'Year {year_num}'

    # 모델명을 컬럼 제목으로 변환
    paper_model_names = []
    for i, name in enumerate(model_names):
        paper_model_names.append(f"({i+1})\n{name}")

    # 추가 메타정보 행
    n_models = len(models)

    add_rows = {
        "Firm Controls": ["No"] + ["Yes"] * (n_models - 1),
        "Market Controls": ["No"] * min(2, n_models) + ["Yes"] * max(0, n_models - 2),
        "Year Fixed Effects": ["No"] * min(3, n_models) + ["Yes"] * max(0, n_models - 3),
        "Standard Errors": ["Robust"] * n_models
    }

    # 실제 모델 수에 맞게 조정
    for key in add_rows:
        add_rows[key] = add_rows[key][:n_models]

    # 표 생성
    table = make_regression_table(
        results_list=models,
        model_names=paper_model_names,
        var_order=var_order,
        var_rename=var_rename,
        add_rows=add_rows,
        show_adj_r2=True
    )

    # 모든 형식으로 저장
    save_all_formats(table, "sentiment_mbv_regression")

    return table

def final_conclusion(model, df):
    """최종 결론 및 해석"""

    print("\n9. 최종 결론")
    print("="*80)

    if 'sentiment_mean_scaled' not in model.params.index:
        print("분석 실패: 감성 변수가 최종 모델에 포함되지 않음")
        return False

    coef = model.params['sentiment_mean_scaled']
    pval = model.pvalues['sentiment_mean_scaled']
    tstat = model.tvalues['sentiment_mean_scaled']
    conf_int = model.conf_int().loc['sentiment_mean_scaled']

    print(f"핵심 가설검정 결과:")
    print(f"H1: 뉴스 감성점수가 높을수록 기업가치(MBV)가 높다")
    print(f"")
    print(f"통계적 결과:")
    print(f"   - 회귀계수(β): {coef:.6f}")
    print(f"   - t-통계량: {tstat:.4f}")
    print(f"   - p-value: {pval:.6f}")
    print(f"   - 95% 신뢰구간: [{conf_int[0]:.6f}, {conf_int[1]:.6f}]")
    print(f"   - 표본 크기: {int(model.nobs)}")
    print(f"   - R-squared: {model.rsquared:.4f}")

    # 통계적 유의성 판정
    if pval < 0.01:
        significance = "1% 수준에서 통계적으로 유의"
        conclusion = "H1 강력 지지"
        success = True
    elif pval < 0.05:
        significance = "5% 수준에서 통계적으로 유의"
        conclusion = "H1 지지"
        success = True
    elif pval < 0.10:
        significance = "10% 수준에서 통계적으로 유의"
        conclusion = "H1 약한 지지"
        success = True
    else:
        significance = "통계적으로 유의하지 않음"
        conclusion = "H1 기각"
        success = False

    print(f"")
    print(f"가설검정 결론:")
    print(f"   - 통계적 유의성: {significance}")
    print(f"   - 연구 결론: {conclusion}")

    # 경제적 해석 (유의한 경우만)
    if success:
        print(f"")
        print(f"경제적 해석:")

        # 표준화 계수 해석
        sentiment_std = df['sentiment_mean'].std()
        mbv_std = df['MBV'].std()

        print(f"   - 감성점수 1 표준편차({sentiment_std:.3f}) 증가 시")
        print(f"   - MBV가 {coef:.4f} 증가 (표준화 기준)")

        # 실제 단위 해석
        actual_effect = coef * mbv_std / sentiment_std
        print(f"   - 실제 단위: 감성점수 0.1 증가 → MBV {actual_effect*0.1:.4f} 증가")

        if coef > 0:
            print(f"   - 해석: 긍정적 뉴스 감성이 기업가치를 향상시킴")
        else:
            print(f"   - 해석: 부정적 결과 (이론과 불일치)")

    print(f"")
    print(f"연구의 한계:")
    print(f"   - 표본 크기: {int(model.nobs)}개 (추가 확대 권장)")
    print(f"   - 내생성: 역인과관계 가능성")
    print(f"   - 생략변수: 관찰되지 않은 기업 특성")

    return success

def main_analysis():
    """메인 분석 실행 - 모델 결과 반환하도록 수정"""

    print("정교한 통계 분석 시작")
    print("="*100)

    try:
        # 1. 데이터 로딩 및 탐색
        news_df, financial_df, market_df = load_and_explore_data()

        # 2. 데이터 정제
        news_clean, financial_clean, market_clean = clean_and_standardize_data(
            news_df, financial_df, market_df
        )

        # 3. 변수 생성
        sentiment_features = create_sentiment_aggregates(news_clean)
        financial_features = create_financial_ratios(financial_clean)

        # 4. 데이터 병합
        final_dataset = merge_all_data(sentiment_features, financial_features, market_clean)

        # 표본 크기 확인
        if len(final_dataset) < 50:
            print(f"경고: 표본 크기가 부족합니다 (N={len(final_dataset)})")
            print("최소 50개 이상의 관측치가 권장됩니다.")
            return False, None, None, None

        # 5. 데이터 품질 검증
        validate_data_quality(final_dataset)

        # 6. 회귀분석 실행 (모델들 반환)
        models, model_names, best_model = run_hierarchical_regression(final_dataset)

        if models is None:
            print("회귀분석 실행 실패")
            return False, None, None, None

        # 7. 논문용 표 생성 및 저장
        table = create_paper_table(models, model_names)

        # 8. 최종 결론
        success = final_conclusion(best_model, final_dataset)

        print(f"\n" + "="*100)
        if success:
            print("분석 성공: 통계적으로 유의한 결과 도출")
            print("뉴스 감성이 기업가치에 미치는 영향 실증적으로 확인")
        else:
            print("분석 완료: 통계적 유의성 미달")
            print("가설 지지 증거 부족 - 추가 연구 필요")

        print("\n파일 생성 완료:")
        print("- sentiment_mbv_regression.xlsx (Excel 표)")
        print("- sentiment_mbv_regression.csv (CSV 표)")
        print("- sentiment_mbv_regression.tex (LaTeX 코드)")
        print("="*100)

        return success, models, best_model, table

    except Exception as e:
        print(f"분석 중 오류 발생: {str(e)}")
        import traceback
        traceback.print_exc()
        return False, None, None, None

# 분석 실행
if __name__ == "__main__":
    success, models, best_model, table = main_analysis()

    if success:
        print("\n연구 성과: 학술 논문 작성 및 정책 제언 가능")
        print("신뢰할 수 있는 실증 결과 확보")
        print("논문용 표가 자동으로 생성되었습니다.")
    else:
        print("\n연구 개선 방향:")
        print("1. 표본 크기 확대")
        print("2. 추가 통제변수 도입")
        print("3. 내생성 문제 해결 (도구변수 등)")
        print("4. 산업별 세분화 분析")
        print("5. 시간 지연 효과 검토")

정교한 통계 분석 시작

1. 데이터 로딩 및 기초 탐색
--------------------------------------------------------------------------------
뉴스 데이터: (12789, 18)
재무 데이터: (18701, 11)
시가총액 데이터: (21892, 9)

감성점수 분포:
평균: 0.2927, 표준편차: 0.3192
범위: [-0.771, 0.755]

2. 데이터 정제 및 표준화
--------------------------------------------------------------------------------
2-1. 뉴스 데이터 정제
   정제 후: (12745, 20)
2-2. 재무 데이터 정제
   정제 후: (16793, 12)
2-3. 시가총액 데이터 정제
   정제 후: (5355, 6)

3. 뉴스 감성 집계 (기업-연도별)
--------------------------------------------------------------------------------
   집계 결과: 1947 → 581 (뉴스 5개 이상)
   평균 뉴스 개수: 18.0

4. 재무 비율 및 지표 생성
--------------------------------------------------------------------------------
   재무 지표 생성 완료: (16793, 23)

5. 데이터 병합 및 MBV 계산
--------------------------------------------------------------------------------
   재무-시장 병합: (15046, 27)
   최종 병합: (406, 40)
   MBV 계산 완료: (396, 43)
   MBV 분포: 평균=2.215, 중위값=1.766
   MBV 범위: [0.271, 10.181]

6. 데이터 품질 검증
-------------------------------------------

In [55]:
# 1. 데이터 로딩 및 탐색
news_df, financial_df, market_df = load_and_explore_data()

# 2. 데이터 정제
news_clean, financial_clean, market_clean = clean_and_standardize_data(
    news_df, financial_df, market_df
)

# 3. 변수 생성
sentiment_features = create_sentiment_aggregates(news_clean)
financial_features = create_financial_ratios(financial_clean)

# 4. 데이터 병합
final_dataset = merge_all_data(sentiment_features, financial_features, market_clean)
final_dataset


1. 데이터 로딩 및 기초 탐색
--------------------------------------------------------------------------------
뉴스 데이터: (12789, 18)
재무 데이터: (18701, 11)
시가총액 데이터: (21892, 9)

감성점수 분포:
평균: 0.2927, 표준편차: 0.3192
범위: [-0.771, 0.755]

2. 데이터 정제 및 표준화
--------------------------------------------------------------------------------
2-1. 뉴스 데이터 정제
   정제 후: (12745, 20)
2-2. 재무 데이터 정제
   정제 후: (16793, 12)
2-3. 시가총액 데이터 정제
   정제 후: (5355, 6)

3. 뉴스 감성 집계 (기업-연도별)
--------------------------------------------------------------------------------
   집계 결과: 1947 → 581 (뉴스 5개 이상)
   평균 뉴스 개수: 18.0

4. 재무 비율 및 지표 생성
--------------------------------------------------------------------------------
   재무 지표 생성 완료: (16793, 23)

5. 데이터 병합 및 MBV 계산
--------------------------------------------------------------------------------
   재무-시장 병합: (15046, 27)
   최종 병합: (406, 40)
   MBV 계산 완료: (396, 43)
   MBV 분포: 평균=2.215, 중위값=1.766
   MBV 범위: [0.271, 10.181]


,자산총계,부채총계,자본총계,매출액,당기순이익,bsns_year,reprt_code,report_name,report_date,기업명,...,negative_mean,negative_std,confidence_mean,confidence_std,sentiment_volatility,confidence_weighted_sentiment,news_intensity,MBV,log_MBV,market_liquidity
0,1.072148e+11,3.938808e+10,6.782669e+10,2.463637e+09,-3.684338e+09,2024,11012,반기보고서,2024-06-30,HLB파나진,...,0.396014,0.158175,0.626757,0.137438,0.316339,0.130347,0.019178,1.964762,0.675371,19.466385
1,9.154201e+10,2.873830e+10,6.280371e+10,3.407320e+09,5.712507e+08,2024,11013,1분기보고서,2024-03-31,HLB파나진,...,0.396014,0.158175,0.626757,0.137438,0.316339,0.130347,0.019178,2.121902,0.752313,19.466385
2,1.096106e+11,3.348303e+10,7.612757e+10,1.319692e+10,-2.085569e+09,2024,11011,사업보고서,2024-12-31,HLB파나진,...,0.396014,0.158175,0.626757,0.137438,0.316339,0.130347,0.019178,1.750526,0.559917,19.466385
3,1.067624e+11,3.418568e+10,7.257672e+10,3.851908e+09,-1.222960e+08,2024,11014,3분기보고서,2024-09-30,HLB파나진,...,0.396014,0.158175,0.626757,0.137438,0.316339,0.130347,0.019178,1.836172,0.607683,19.466385
4,5.648974e+11,4.037330e+11,1.611644e+11,4.584978e+11,1.730768e+10,2021,11011,사업보고서,2021-12-31,HS애드,...,0.386757,0.037370,0.613243,0.037370,0.074711,0.138891,0.019178,0.665790,-0.406781,19.133117
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
401,5.290252e+10,8.183910e+09,4.471861e+10,1.455840e+10,-9.203877e+09,2024,11013,1분기보고서,2024-03-31,하이퍼코퍼레이션,...,0.356000,0.103032,0.645160,0.101334,0.206065,0.185802,0.082192,2.945535,1.080290,20.744028
402,5.484090e+10,9.220361e+09,4.562053e+10,1.807624e+10,1.034505e+09,2024,11012,반기보고서,2024-06-30,휴림에이텍,...,0.270500,0.091305,0.729500,0.091305,0.182609,0.334841,0.013699,0.761693,-0.272212,20.564144
403,5.383891e+10,9.107988e+09,4.473092e+10,1.782476e+10,1.315785e+09,2024,11013,1분기보고서,2024-03-31,휴림에이텍,...,0.270500,0.091305,0.729500,0.091305,0.182609,0.334841,0.013699,0.776841,-0.252519,20.564144
404,7.531147e+10,2.739607e+10,4.791540e+10,7.031696e+10,4.597622e+09,2024,11011,사업보고서,2024-12-31,휴림에이텍,...,0.270500,0.091305,0.729500,0.091305,0.182609,0.334841,0.013699,0.725212,-0.321291,20.564144


In [58]:
final_dataset.columns

Index(['자산총계', '부채총계', '자본총계', '매출액', '당기순이익', 'bsns_year', 'reprt_code',
       'report_name', 'report_date', '기업명', '종목코드', 'year', 'debt_to_equity',
       'debt_to_assets', 'roa', 'roe', 'log_assets', 'log_equity',
       'has_revenue', 'asset_turnover', 'asset_growth', 'equity_growth',
       'size_quartile', 'market_cap_krw', 'shares_outstanding', 'volume',
       'value_traded', 'sentiment_mean', 'sentiment_std', 'news_count',
       'sentiment_median', 'positive_mean', 'positive_std', 'negative_mean',
       'negative_std', 'confidence_mean', 'confidence_std',
       'sentiment_volatility', 'confidence_weighted_sentiment',
       'news_intensity', 'MBV', 'log_MBV', 'market_liquidity'],
      dtype='object')

In [59]:
# 변수 표준화
scaler = StandardScaler()

continuous_vars = ['sentiment_mean', 'log_assets', 'debt_to_equity', 'roa', 'market_liquidity']
available_vars = [var for var in continuous_vars if var in final_dataset.columns]

for var in available_vars:
    final_dataset[f'{var}_scaled'] = scaler.fit_transform(final_dataset[[var]])

# 연도 더미 (중요한 연도만)
year_counts = final_dataset['year'].value_counts()
major_years = year_counts[year_counts >= 10].index  # 관측치 10개 이상 연도만

for year in major_years[1:]:  # 첫 번째 연도는 기준
    final_dataset[f'year_{year}'] = (final_dataset['year'] == year).astype(int)

print(f"분석 준비 완료: {final_dataset.shape}")
print(f"사용 가능한 표준화 변수: {[var for var in final_dataset.columns if var.endswith('_scaled')]}")

분석 준비 완료: (396, 51)
사용 가능한 표준화 변수: ['sentiment_mean_scaled', 'log_assets_scaled', 'debt_to_equity_scaled', 'roa_scaled', 'market_liquidity_scaled']


#### 테이블 내보내기

In [64]:
pd.DataFrame(final_dataset[['log_assets_scaled','sentiment_mean_scaled','debt_to_equity_scaled',
                                                'log_assets_scaled','market_liquidity_scaled','MBV','log_MBV']].corr(numeric_only=True))

,log_assets_scaled,sentiment_mean_scaled,debt_to_equity_scaled,log_assets_scaled,market_liquidity_scaled,MBV,log_MBV
log_assets_scaled,1.000000,0.025306,0.280660,1.000000,0.238695,-0.282134,-0.292856
sentiment_mean_scaled,0.025306,1.000000,-0.116460,0.025306,0.096581,-0.047332,0.000828
debt_to_equity_scaled,0.280660,-0.116460,1.000000,0.280660,-0.009666,0.178069,0.160269
log_assets_scaled,1.000000,0.025306,0.280660,1.000000,0.238695,-0.282134,-0.292856
market_liquidity_scaled,0.238695,0.096581,-0.009666,0.238695,1.000000,0.237900,0.283077
MBV,-0.282134,-0.047332,0.178069,-0.282134,0.237900,1.000000,0.908508
log_MBV,-0.292856,0.000828,0.160269,-0.292856,0.283077,0.908508,1.000000


In [66]:
pd.DataFrame(final_dataset[['log_assets_scaled','sentiment_mean_scaled','debt_to_equity_scaled',
                                                'log_assets_scaled','market_liquidity_scaled','MBV','log_MBV']].corr(numeric_only=True)).to_csv("TABLE2_가설1_최종검정때변수_상관관계행렬.csv",encoding='utf-8-sig')